In [ ]:
# Allow PyTorch to grow GPU memory in expandable chunks, which helps avoid out-of-memory errors from fragmentation
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Part 1 : Transforming XML files to CSV files

In [ ]:
# Connect Google Drive to the Colab session so we can read and save files there
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# Download the small Norwegian spaCy language model, used later for text statistics on the extracted essays
!python -m spacy download nb_core_news_sm

In [ ]:
# Importing Path to handle filesystem paths
from pathlib import Path

In [ ]:
# Set the path to the folder in Google Drive that holds the XML files
ask_dir = Path('/content/drive/MyDrive/xml')

In [ ]:
# Checking if the directionary actually exists
ask_dir, ask_dir.exists()

In [ ]:
# Now im going to use the glob module to find all pathnames ending with *.xml, showing the first five entires and counting all total entires (it should output 1935 entries)
list(ask_dir.glob('*.xml'))[:5], len(list(ask_dir.glob('*.xml')))

In [ ]:
import xml.etree.ElementTree as ET # for parsing XML data
import pandas as pd # for building the dataframe later
import re # for reconstructing essays later (if necessary)
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

In [ ]:
# Im going to choose one path to experiment with
one_path = sorted(list(ask_dir.glob('*.xml')))[0] # the first file in the folder ending with xml

one_path # printing out the path, it should output h0001.xml

In [ ]:
# Parsing it
tree = ET.parse(one_path)
root = tree.getroot()

In [ ]:
person = root.find('.//particDesc/person') # finding the first 'person' element that is a direct child of 'PartiDesc'
print("Person is None:", person is None)

if person is not None:
  print("Number of <p> children:", len(person.findall('p')))
  for p in person.findall('p'):
    print(p.get('n'), '->', repr(p.text))


In [ ]:
# Function to extract person metadata
def extract_person(root): # making the definition
  metadata = {} # Initiating a dictionary

  person = root.find('.//particDesc/person')
  if person is not None:
    for p in person.findall('p'):
      key = p.get('n')
      value = p.text
      metadata[key] = value

  return metadata

metadata = extract_person(root) # testing on root
metadata



In [ ]:
# Rebuild each essay's full text from the XML.
# Walks through every <word> inside the <body> tag and joins them into one string.
# If use_norm is True it uses the corrected spelling; otherwise it uses the original text.

def extract_essay(root, use_norm=False): # Defining a function to extract the essays, use_norm is initially false, meaning using the normalized/corrected word in the essay is initially false
  body = root.find(".//body") # We're finding the first 'body' element which is a direct child of previous tags
  if body is not None:
    essay = []
    for word in body.iterfind(".//word"):
      token = None

      if use_norm:
        token = word.get('norm')
      if not token:
          token = (word.text or "").strip()
      if token:
          essay.append(token)
    if not essay:
      return ""

    text= " ".join(essay)

    return text

In [ ]:
# Extract one essay using the original (uncorrected) text, then print the first 400 characters as a quick check
text_original = extract_essay(root, use_norm=False)
print(text_original[:400])

In [ ]:
# Extract one essay using the corrected (normalized) text, then print the first 400 characters as a quick check
text_normalized = extract_essay(root, use_norm=True)
print(text_normalized[:400])

In [ ]:
# Extract the front matter of an essay: the title and the task prompt.
# Looks inside the <front> tag, grabs the title, then digs into the second <div>
# (which holds the prompt) to pull out its heading and paragraph text.

def extract_prompt(root):

  data = {'title' : None,
          'prompt' : None,
          'text' : None
          }


  front = root.find('.//front')
  if front is None:
    return data


  title_div = front.find(".//div[@type = 'title']")
  if title_div is not None and title_div.text: # If title_div is not None/has a title and there is some text in the belonging text element, we assign the title
    data['title'] = title_div.text # Assigning the title to the text element

  other_div = front.findall('div')
  if len(other_div) > 1: # if the amount of other divs is more than 1, we assign the variable div to the second div found which is where the title and prompt lies
    div = other_div[1] # div containing title and task prompt
    head = div.find('head') # finding the head (title) element
    if head is not None:
      data["prompt_context"] = "".join(head.itertext()).strip()


    p = div.find('p') # task prompt of the each essay
    if p is not None:
      data['prompt_task'] = "".join(p.itertext()).strip()


  return data

In [ ]:
# testing the prompt
prompt_info = extract_prompt(root)

prompt_info

In [ ]:
# Main loop: go through every XML file, extract its metadata, essay text, and prompt,
# and collect each as a row. Successful rows go into 'rows'; any file that fails goes into 'errors'.
# At the end, build a pandas DataFrame from all the collected rows.

rows = [] # defining a list to contain all my rows

errors = [] # defining a list to contain all my errors

# Here I define all my xml_files
xml_files = sorted(list(ask_dir.glob('*.xml')))

# Step one is using enumerate to loop over the xml files
for index, path in enumerate(xml_files, start=1):

  if index == 1 or index % 100 == 0 or index == len(xml_files):
    print(f"Processing file {index}/{len(xml_files)}: {path.name}") # Printing out progress for every 100 files looped over


  try:
    tree = ET.parse(path)
    root = tree.getroot()

    file_id = path.stem # path identifier

    meta = extract_person(root) # Defining a new variable to hold metadata using our function

    text_original = extract_essay(root, use_norm=False) # Essay texts extraction, defining two variables to use both the original and the normalized one

    text_normalized = extract_essay(root, use_norm=True) # use_norm is true here because we're using the normalized/corrected words

    prompt_info = extract_prompt(root) # Storing our prompt info

    row = {
        'file_id': file_id,
        'text_original': text_original,
        'text_normalized': text_normalized,
    }

    row.update(meta)
    row.update(prompt_info)

    rows.append(row)

  except Exception as e:
    errors.append(path.name, repr(e))
    print(f"error in {path.name}: {e}")

print(f"Processed {len(rows)} files.")
if errors:
  print(f"{len(errors)} files had errors.")


df = pd.DataFrame(rows) # Here we finally use pandas to turn the rows into a dataframe
print(df.shape)
df.head() # To see the first five entries

In [ ]:
# Printing it out to CSV for saving
csv_path = ask_dir / 'ask_master.csv'
df.to_csv(csv_path, index=False, encoding='utf-8')


# Part 2 : Data analysis on the CSV file

In [ ]:
# Here I'm defining the path for the master csv file
csv_path = '/content/drive/MyDrive/ask_master.csv' # the path to the file in my drive

df = pd.read_csv(csv_path) # using pandas method to read the csv file
df.head() # inspecting the first five entries of the dataframe


In [ ]:
# I'll do some NLP preprocessing with SpaCy / Text enrichment with SpaCy
import spacy

nlp = spacy.load('nb_core_news_sm') # Loading the model

import nb_core_news_sm

nlp = nb_core_news_sm.load() # Loading the model


In [ ]:
# Making a function that uses the nlp function on a text in text_original for statistics

def preprocess_text(text):
  doc = nlp(text)

  tokens = [token for token in doc if not token.is_space] # For individual tokens, is_space checks if token is whitespace

  words = [word for word in doc if word.is_alpha] # For individual words, is_alpha checks if token text consists of alphabetic characters

  sentences = [sentence for sentence in doc.sents] # For individual sentences

  return len(words), len(sentences), len(tokens) # We're also counting the document to get the total number of tokens

# Testing it out on all the rows in 'text_normalized'
counts = df['text_original'].apply(preprocess_text)

# Using pd.Series to sum it up
total_counts = counts.apply(pd.Series).sum()

# Giving the columns names for readability
total_words, total_sentences, total_tokens = total_counts.values

print(f'Total words: {total_words}')
print(f'Total sentences: {total_sentences}')
print(f'Total tokens: {total_tokens}')

In [ ]:
# Before proceeding to doing any missing values treatment, I'll just do some quick analysis over how the average essay text length

'''
I'll start by making the counts into a dataframe so i can use mean() to find the average per row
'''

counts_df = counts.apply(pd.Series)
counts_df.columns = ['words', 'sentences', 'tokens']

total_words = counts_df['words'].sum()
total_sentences = counts_df['sentences'].sum()
total_tokens = counts_df['tokens'].sum()

'''
Now to find all the averages, I will simply use the function mean()
'''

avg_words = counts_df['words'].mean()
avg_sentences = counts_df['sentences'].mean()
avg_tokens = counts_df['tokens'].mean()

'''
I also want to find all the minimum and maximum of each essay
'''
min_words = counts_df['words'].min()
max_words = counts_df['words'].max()

print(f'Average words per essay: {avg_words:.2f}')
print(f'Average sentences per essay: {avg_sentences:.2f}')
print(f'Average tokens per essay: {avg_tokens:.2f}')
print(f'Minimum words in an essay: {min_words:.2f}')
print(f'Maximum words in an essay: {max_words:.2f}')

In [ ]:
counts_df.describe().T

In [ ]:
df.info() # Here I want information regarding how many columns and datatype(s) are in df

In [ ]:
# Replace corpus-specific missing symbols with NaN
df_clean = df.replace(['-', '.'], np.nan)

In [ ]:
missing_n = df_clean.isna().sum()

missing_pct = df_clean.isna().mean() * 100

missing_table = pd.DataFrame({
    'Missing (N)': missing_n,
    'Missing (%)': missing_pct.round(2)
})

# Sort from most missing → least missing
missing_table = missing_table.sort_values('Missing (%)', ascending=False)

missing_table

In [ ]:
df['tokens'] = counts_df['tokens']
df['words'] = counts_df['words']
df['sentences'] = counts_df['sentences']

While there are many columns worth exploring here, I'll be looking at the ones that are relevant for RQ1 (Text_original and CEFRscore) and RQ2 (Country, language, age, gender occupation).

In [ ]:
# List of variables retained for automated grading and bias evaluation
retained_columns = [
    "prompt_task",
    "text_original",
    "CEFRscore",
    "age",
    "gender",
    "country",
    "language",
    "occupation"
]

# Create a copy of the dataset with only the retained variables
df= df[retained_columns].copy()

# Check shape
print("Subset shape:", df.shape)

# Preview missing values
print(df.isna().sum())

In [ ]:
df['country'].unique() # Checking the unique values in the country column ''Disse kan også binnes, inn i lande-grupper'

In [ ]:
# I need to see how many are encoded as '.'
(df['country'] == '.').mean() * 100

In [ ]:
# Since the number of '.' is relatively small, i'll remove these entries
df = df.drop(df[df['country'] == '.'].index)

In [ ]:
# Binning countries based on SSBs landkategorier when it comes to immigration
'''
Here I'm going to do the same as I did with occupation with countries
'''

ssb_country_map = {
    #EU-land i Øst-Europa
    'Polen': 'EU-land i Øst-Europa',
    'Litauen': 'EU-land i Øst-Europa',
    'Latvia': 'EU-land i Øst-Europa',
    'Estland': 'EU-land i Øst-Europa',

    #Øst-Europa ellers
    'Bosnia-Hercegovina': 'Øst-Europa ellers',
    'Jugoslavia': 'Øst-Europa ellers',
    'Kroatia': 'Øst-Europa ellers',
    'Albania': 'Øst-Europa ellers',
    'Hviterussland': 'Øst-Europa ellers',
    'Russland': 'Øst-Europa ellers',
    'Ukraina': 'Øst-Europa ellers',
    'Kosovo': 'Øst-Europa ellers',
    'Makedonia': 'Øst-Europa ellers',
    'Serbia-Montenegro': 'Øst-Europa ellers',
    'Moldova': 'Øst-Europa ellers',

    #Nord-Amerika
    'USA': 'Nord-Amerika og Oseania',
    'Canada': 'Nord-Amerika og Oseania',
    'Australia': 'Nord-Amerika og Oseania',
    'New Zealand': 'Nord-Amerika og Oseania',

    #Sør og Mellom-Amerika
    'Cuba': 'Sør- og Mellom-Amerika',
    'Mexico': 'Sør- og Mellom-Amerika',
    'Guatemala': 'Sør- og Mellom-Amerika',
    'Venezuela': 'Sør- og Mellom-Amerika',
    'Chile': 'Sør- og Mellom-Amerika',
    'Argentina': 'Sør- og Mellom-Amerika',
    'Dominikanske Republikk': 'Sør- og Mellom-Amerika',
    'Colombia': 'Sør- og Mellom-Amerika',
    'Peru': 'Sør- og Mellom-Amerika',
    'Ecuador': 'Sør- og Mellom-Amerika',
    'Bolivia': 'Sør- og Mellom-Amerika',
    'Uruguay': 'Sør- og Mellom-Amerika',
    'El Salvador': 'Sør- og Mellom-Amerika',
    'Barbados': 'Sør- og Mellom-Amerika',
    'Nicaragua': 'Sør- og Mellom-Amerika',
    'Costa Rica': 'Sør- og Mellom-Amerika',
    'Surinam': 'Sør- og Mellom-Amerika',
    'Guyana': 'Sør- og Mellom-Amerika',

    #Asia
    'Pakistan': 'Asia',
    'Vietnam': 'Asia',
    'Israel': 'Asia',
    'Indonesia': 'Asia',
    'Thailand': 'Asia',
    'Khazakhstan': 'Asia',

    #Afrika
    'Mauritius': 'Afrika',
    'Sør-Afrika': 'Afrika',
    'Zimbabwe': 'Afrika',
    'Somalia': 'Afrika',
    'Nigeria': 'Afrika',
    'Sierra Leone': 'Afrika',
    'Ghana': 'Afrika',
    'Liberia': 'Afrika',
    'Kamerun': 'Afrika',
    'Kenya': 'Afrika',
    'Etiopia': 'Afrika',

    #Vest-Europa, not a part of the SBB regions above, but I'll include them as a West Europe to not lose them
    'Norge': 'Vest-Europa',
    'Nederland': 'Vest-Europa',
    'Belgia': 'Vest-Europa',
    'Spania': 'Vest-Europa',
    'Tyskland': 'Vest-Europa',
    'Østerrike': 'Vest-Europa',
    'Irland': 'Vest-Europa',
    'Malta': 'Vest-Europa',
    'Sveits/Liechtenstein': 'Vest-Europa',
    'Storbritannia': 'Vest-Europa'

}

df['country_ssb_region'] = df['country'].map(ssb_country_map)

In [ ]:
# lets first check if there are NaN values in country ssb regions
df['country_ssb_region'].isna().sum()

In [ ]:
# I would like to visualise the country_ssb_region to see if the classes are balanced
import matplotlib.pyplot as plt

country_counts = df['country_ssb_region'].value_counts()

plt.figure(figsize=(12, 6))

country_counts.plot(kind='bar', edgecolor='black')

plt.xlabel('SSB region')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.margins(x=0.05)
plt.tight_layout()
plt.show()

In [ ]:
df['language'].unique() # Checking the unique values in the language column

In [ ]:
# I want to check how many language rows have an empty CEFR row
df[df['language'].isna()]['CEFRscore'].isna().sum()

In [ ]:
df[df['CEFRscore'].isna()]['language'].value_counts(dropna=False)

In [ ]:
#For the bias evaluation, I need to remove all the languages that do not have an CEFR score
df = df.dropna(subset=['CEFRscore'])

print(df['language'].value_counts(dropna=False))
print(df.shape)

In [ ]:
df['age'].unique() # Simply just checking what the unique values are here

In [ ]:
# I need to check the percentage of entries encoded as .
(df['age'] == '.').mean() * 100

In [ ]:
# Checking what kind of rows are encoded with '.' to check if the missingness seems to be at random (MAR)
df[df['age'] == '.'] # Inspecting the first five entries


In [ ]:
# After checking, I see that the missingness seems to be at random (MAR) as some women don't have the age listed and some men don't. I'll remove these columns encoded with '.'
df = df.drop(df[df['age'] == '.'].index)


In [ ]:
# Checking again the unique values of df['age']
df['age'].unique()

# Converting these ages to int64
df['age'] = df['age'].astype('int64')

In [ ]:
# making an age distribution plot, but im going to bin the ages together as based on SSB age bins
import matplotlib.pyplot as plt

age_bins = [0, 25, 40, 55, float('inf')]

labels = ['Under 25', '25–39', '40–54', '55 og eldre'] # Loosely based on SSB bins

df['age_group'] = pd.cut(df['age'],
                         bins=age_bins,
                         labels=labels,
                         right=False) # Using pdf.cut to define my groups



plt.figure(figsize=(10, 7))

age_count = df['age_group'].value_counts().sort_index()

age_count.plot(kind='bar', edgecolor='black', color='hotpink')  #  plotting the graph

plt.xlabel('Age group')
plt.xticks(rotation=0)
plt.ylabel('Count')
plt.title('Age Distribution (SSB groups)')
plt.tight_layout()
plt.show()

In [ ]:
# I want to do the same for gender as I will use it for my RQs.
df['gender'].value_counts()

In [ ]:
# Checking the percentage of missing values (.) in gender
(df['gender'] == '.').mean() * 100

In [ ]:
# Since there is one dot, presumably someone who didn't want to disclose their gender, I will remove it from the dataframe
df = df.drop(df[df['gender'] == '.'].index)

In [ ]:
# Making a gender plot
import matplotlib.pyplot as plt

gender_counts = df['gender'].value_counts()

gender_counts.plot(kind='bar', color = 'teal', edgecolor='black')

plt.xticks(rotation=0)

plt.show()

In [ ]:
# Here i'm going to make a stacked histogram to show age distribution by gender
import numpy as np

counts = pd.crosstab(df['age_group'], df['gender']).reindex(labels)

plt.figure(figsize=(10,7))
counts.plot(kind='bar', stacked=True, edgecolor='black', color=['hotpink', 'lightblue'], figsize=(10,7))

plt.xlabel('Age group')
plt.xticks(rotation=0)
plt.ylabel('Count')
plt.title('Age Distribution by Gender (SSB age groups)')
plt.legend(title='Gender')
plt.tight_layout()
plt.show()

In [ ]:
df['occupation'].unique() # Disse også binnes for å unngå korrelasjoner. (correlations) Vi binner fordi vi ser på gruppeindivid ->

In [ ]:
# Similar to country, here I want to check the percentages of '.' values
(df['occupation'] == '.').mean() * 100

In [ ]:
# Here I want to remove the '.' coded variables, but this also does result in significant loss of dataentry
df = df.drop(df[df['occupation'] == '.'].index)

In [ ]:
# Group the many individual occupation labels into broader SSB-style categories.
# Build a dictionary mapping each raw occupation to its group, then apply it
# to create a new 'occupation_ssb_group' column.

'''
First, I'll create a map based on the existing occupations, mapping them to their occupations to corresponding occupation groups
'''

ssb_map = {
    'servicenæring': 'Service og salg',
    'manuelt arbeid': 'Håndverk og manuelt',
    'opplæring/undervisning': 'Akademiske profesjoner',
    'lege': 'Akademiske profesjoner',
    'kontorarbeid': 'Kontor og administrasjon',
    'hjelpepleier': 'Service og omsorg',
    'forskning': 'Akademiske profesjoner',
    'sykepleier': 'Service og omsorg',
    'akademisk yrke': 'Akademiske profesjoner',
    'politi, toll, brann': 'Annet',
    'transport': 'Annet',
    'kultur': 'Akademiske profesjoner',
    'andre akademiske yrker': 'Akademiske profesjoner',
    'helsearbeid': 'Service og omsorg',
    'student': 'Utenfor arbeidsstyrken',
    'hjemmeværende': 'Utenfor arbeidsstyrken',
    'privat næringsliv': 'Annet',
    'annet': 'Annet',
}

df['occupation_ssb_group'] = df['occupation'].map(ssb_map)

In [ ]:
# checking NaN in occupation_ssb_group
df['occupation_ssb_group'].isna().sum()

In [ ]:
# Here I also want to visualize the various groups in occupation_ssb_group
import matplotlib.pyplot as plt

occupation_counts = df['occupation_ssb_group'].value_counts()


plt.figure(figsize=(12, 6))

occupation_counts.plot(kind='bar', edgecolor='black')

plt.xlabel('SSB occupation')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.margins(x=0.05)
plt.tight_layout()
plt.show()

In [ ]:
df['prompt_task'].unique() # Checking the unique values

In [ ]:
# I want to check how many entries in prompt_task are empty
(df['prompt_task'] == '').sum()

In [ ]:
df['CEFRscore'].unique() # Checking the unique values

In [ ]:
df['CEFRscore'].value_counts()

In [ ]:
df['CEFRscore'].isna().sum() # this is an incredibly cruicial column, so I need to know how many are nan


In [ ]:

cefr_counts = df['CEFRscore'].value_counts().sort_index()

plt.figure(figsize=(8,5))
cefr_counts.plot(kind='bar', color='lightpink', edgecolor='black')

plt.xlabel('CEFR level')
plt.xticks(rotation=0)
plt.ylabel('Count')
plt.title('Distribution of CEFR Scores')
plt.tight_layout()
plt.show()

In [ ]:
df = df[df['CEFRscore'].notna()] # Removing all of the missing values from CEFRscore column

In [ ]:
# CEFR labels
cefr_labels = ['A2', 'A2/B1', 'B1', 'B1/B2', 'B2', 'B2/C1', 'C1']

# I want to see whether I can see a pattern when it comes to gender distribution in the CEFR scores
counts = pd.crosstab(df['CEFRscore'], df['gender']).reindex(cefr_labels)


plt.figure(figsize=(10,7))
counts.plot(kind='bar', stacked=True, edgecolor='black', color=['hotpink', 'lightblue'], figsize=(10,7))

plt.xlabel('CEFR grades')
plt.xticks(rotation=0)
plt.ylabel('Count')
plt.title('Gender Distribution by CEFR scores')
plt.legend(title='Gender')
plt.tight_layout()
plt.show()

In [ ]:
# I want to see the mean words, median, min, max and count in of the CEFR target essays
df['words'] = df['text_original'].apply(lambda x: preprocess_text(x)[0])

cefr_length_essay = (
    df.groupby('CEFRscore')['words']
      .agg(['mean', 'median', 'min', 'max', 'std','count'])
      .sort_values(by='count', ascending = False)
)

print(cefr_length_essay)

In [ ]:
# Count the words in each essay, then group by region to see essay-length statistics.
# For each region, compute the mean, median, min, max, standard deviation, and count
# of word counts, sorted from longest average essay to shortest.

df['words'] = df['text_original'].apply(lambda x: preprocess_text(x)[0])
region_length = (
    df.groupby('country_ssb_region')['words']
      .agg(['mean', 'median', 'min', 'max', 'std','count'])
      .sort_values(by='mean', ascending=False)
)

print(region_length)

In [ ]:
# Same length statistics as before, but grouped by the writer's language instead of region.
# For each language, compute the mean, median, min, max, standard deviation, and count
# of essay word counts, sorted from longest average essay to shortest.

language_length = (
    df.groupby('language')['words']
      .agg(['mean', 'median', 'min', 'max','std','count'])
      .sort_values(by='mean', ascending=False)
)

print(language_length)

In [ ]:
# Cross-tabulate language against CEFR score to see the proficiency spread within each L1.
# normalize='index' turns the counts into row percentages (each language's scores sum to 100%),
# so you can compare distributions across languages of different sizes.

cefr_by_L1_percent = pd.crosstab(
    df['language'],
    df['CEFRscore'],
    normalize='index'
) * 100

print(cefr_by_L1_percent.round(2))

In [ ]:
# Same CEFR cross-tabulation as before, but broken down by gender instead of language.
# Row percentages (normalize='index') show how CEFR scores are distributed within each gender.

cefr_by_gender = pd.crosstab(
    df['gender'],
    df['CEFRscore'],
    normalize='index'
)    * 100

print(cefr_by_gender.round(2))

In [ ]:
# Same CEFR cross-tabulation again, this time broken down by region.
# Row percentages (normalize='index') show how CEFR scores are distributed within each region.

cefr_by_region = pd.crosstab(
    df['country_ssb_region'],
    df['CEFRscore'],
    normalize='index'
)    * 100

print(cefr_by_region.round(2))

In [ ]:
# Same CEFR cross-tabulation again, this time broken down by age group.
# Row percentages (normalize='index') show how CEFR scores are distributed within each age group.

cefr_by_age_group = pd.crosstab(
    df['age_group'],
    df['CEFRscore'],
    normalize='index'
    )    * 100

print(cefr_by_age_group.round(2))

In [ ]:
# Same CEFR cross-tabulation again, this time broken down by occupation group.
# Row percentages (normalize='index') show how CEFR scores are distributed within each occupation group.

cefr_by_occupation = pd.crosstab(
    df['occupation_ssb_group'],
    df['CEFRscore'],
    normalize='index'
)    * 100

print(cefr_by_occupation.round(2))

In [ ]:
# Counts,how many essays there are per CEFR level and language
counts = pd.crosstab(
    df['CEFRscore'],
    df['language']
)

#Percentages, language distribution within each CEFR level
percent = pd.crosstab(
    df['CEFRscore'],
    df['language'],
    normalize='index'
) * 100

#Adding total N per CEFR level as first column
percent.insert(0, 'N', counts.sum(axis=1))

#round nicely
percent = percent.round(2)

#percent now contains exactly what you need
percent

In [ ]:
# Helper function to build a simple frequency table for any column:
# counts how often each value appears (N) and what share of the total that is (Percent).
# dropna=False means missing values are counted too rather than ignored.

def make_distribution_table(df, column):
    table = (
        df[column]
        .value_counts(dropna=False)
        .to_frame(name='N')
        .assign(Percent=lambda x: (x['N'] / x['N'].sum() * 100).round(2))
    )
    return table

# Build a distribution table for each background variable, then print them all
cefr_table = make_distribution_table(df, 'CEFRscore')
age_table = make_distribution_table(df, 'age_group')
country_table = make_distribution_table(df, 'country_ssb_region')
language_table = make_distribution_table(df, 'language')
occupation_table = make_distribution_table(df, 'occupation_ssb_group')

print(cefr_table)
print(age_table)
print(country_table)
print(language_table)
print(occupation_table)

In [ ]:
# Making a function that uses the nlp function on a text in text_original for statistics

def preprocess_text(text):
  doc = nlp(text)

  tokens = [token for token in doc if not token.is_space] # For individual tokens, is_space checks if token is whitespace

  words = [word for word in doc if word.is_alpha] # For individual words, is_alpha checks if token text consists of alphabetic characters

  sentences = [sentence for sentence in doc.sents] # For individual sentences

  return len(words), len(sentences), len(tokens) # We're also counting the document to get the total number of tokens

# Testing it out on all the rows in 'text_normalized'
counts = df['text_original'].apply(preprocess_text)

# Using pd.Series to sum it up
total_counts = counts.apply(pd.Series).sum()

# Giving the columns names for readability
total_words, total_sentences, total_tokens = total_counts.values

print(f'Total words: {total_words}')
print(f'Total sentences: {total_sentences}')
print(f'Total tokens: {total_tokens}')

In [ ]:
# Before proceeding to doing any missing values treatment, I'll just do some quick analysis over how the average essay text length

'''
I'll start by making the counts into a dataframe so i can use mean() to find the average per row
'''

counts_df = counts.apply(pd.Series)
counts_df.columns = ['words', 'sentences', 'tokens']

total_words = counts_df['words'].sum()
total_sentences = counts_df['sentences'].sum()
total_tokens = counts_df['tokens'].sum()

'''
Now to find all the averages, I will simply use the function mean()
'''

avg_words = counts_df['words'].mean()
avg_sentences = counts_df['sentences'].mean()
avg_tokens = counts_df['tokens'].mean()

'''
I also want to find all the minimum and maximum of each essay
'''
min_words = counts_df['words'].min()
max_words = counts_df['words'].max()

print(f'Average words per essay: {avg_words:.2f}')
print(f'Average sentences per essay: {avg_sentences:.2f}')
print(f'Average tokens per essay: {avg_tokens:.2f}')
print(f'Minimum words in an essay: {min_words:.2f}')
print(f'Maximum words in an essay: {max_words:.2f}')

In [ ]:
counts_df.describe().T

In [ ]:
# Now for testing purposes, I'm going to create a subset of my df with the columns and attributes I need

df_rq1 = df[['text_original', 'prompt_task', 'CEFRscore']]

In [ ]:
df_rq1.head() # Inspecting the first five entries

In [ ]:
'''
Here, we want to make a full factorial for the second research question.
'''

# Build a "full factorial" / counterfactual dataset for RQ2.
# For every essay, generate one row for each possible combination of the five
# background variables (gender, age, region, language, occupation), no matter the
# essay's real values. The row matching the real values is the 'base'; all others
# are 'variant' rows. Each row also records the original value of every variable
# and a flag for which ones were changed, then everything is saved to a parquet file.

import itertools
from tqdm.auto import tqdm

# Defining Df_rq2 with the age group, occupation group, language, and region

df_rq2 = df[['prompt_task', 'text_original', 'age_group', 'gender','language', 'country_ssb_region', 'occupation_ssb_group', 'CEFRscore']]

df_rq2["essay_id"] = range(len(df))

# Collect the set of possible values for each variable, sorted for consistency
contrast_values = {
    "gender": sorted(df_rq2["gender"].unique()),
    "age_group": sorted(df_rq2["age_group"].unique()),
    "country_ssb_region": sorted(df_rq2["country_ssb_region"].unique()),
    "language": sorted(df_rq2["language"].unique()),
    "occupation_ssb_group": sorted(df_rq2["occupation_ssb_group"].unique()),
}


variables = list(contrast_values.keys())

# Every possible combination of the five variables (the Cartesian product)
all_combinations = list(itertools.product(*(contrast_values[v] for v in variables)))

print(f"Group sizes: {{k: len(v) for k, v in contrast_values.items()}}")
print(f"Combinations per essay: {len(all_combinations):,}")
print(f"Essays: {len(df_rq2):,}")
print(f"Total rows: {len(all_combinations) * len(df_rq2):,}")

rows = []

# Expand each essay into one row per combination
for _, row in tqdm(df_rq2.iterrows(), total=len(df_rq2),
                  desc = 'Expanding essays'):
    original_value = {v: row[v] for v in variables}
    essay_dict = row.to_dict()

    for combo in all_combinations:
      combo_dict = dict(zip(variables,combo))

      variant_row = essay_dict.copy()
      variant_row.update(combo_dict)

    # to check which variables are different from the original
      changed_vars = [v for v in variables if combo_dict[v] != original_value[v]]
      n_changed = len(changed_vars)

      variant_row['variant_type'] = 'base' if n_changed == 0 else 'variant'
      variant_row['n_changed'] = n_changed
      variant_row['changed_variable'] = ','.join(changed_vars) if changed_vars else None

    #For per-variable tracking for downstream analysis later on
      for v in variables:
        variant_row[f'{v}_original'] = original_value[v]
        variant_row[f'{v}_is_changed'] = combo_dict[v] != original_value[v]

      rows.append(variant_row)

df_rq2 = pd.DataFrame(rows)

print(f"\nGenerated {len(df_rq2):,} rows across {df_rq2['essay_id'].nunique()} essays")
print(f"Base variants: {(df_rq2['variant_type'] == 'base').sum():,}")
print(f"Counterfactual variants: {(df_rq2['variant_type'] == 'variant').sum():,}")

# Sanity checks
assert (df_rq2.groupby("essay_id").size() == len(all_combinations)).all(), \
    "Not every essay has the expected number of combinations"
assert (df_rq2.groupby("essay_id")["variant_type"].apply(lambda s: (s == "base").sum()) == 1).all(), \
    "Each essay should have exactly one base row"

# Save to disk (parquet is much faster for ~1.76M rows; use csv if you don't have pyarrow)
df_rq2.to_parquet("df_rq2_full_factorial.parquet", index=False)
print("\nSaved to df_rq2_full_factorial.parquet")

In [ ]:
len(# Quick check of the total number of rows in the expanded RQ2 dataframe
len(df_rq2))

In [ ]:
# How many rows, essays, and combinations
print(f"Total rows: {len(df_rq2):,}")
print(f"Unique essays: {df_rq2['essay_id'].nunique()}")
print(f"Rows per essay: {len(df_rq2) // df_rq2['essay_id'].nunique()}")

# Distribution of n_changed
print("\nn_changed distribution:")
print(df_rq2["n_changed"].value_counts().sort_index())

# Spot-check one essay: see that all combinations are present
sample = df_rq2[df_rq2["essay_id"] == 0]
print(f"\nEssay 0 has {len(sample)} rows")
print(f"Base rows: {(sample['variant_type'] == 'base').sum()}")
print(f"Unique (gender, age, region, language, occupation) tuples: "
      f"{sample[['gender','age_group','country_ssb_region','language','occupation_ssb_group']].drop_duplicates().shape[0]}")

In [ ]:
# Preview the first five rows of the expanded RQ2 dataframe
df_rq2.head()

In [ ]:
# Work out how big the full factorial is: how many values each variable has,
# how many combinations that makes per essay, and the total row count across all essays.
from math import prod

sizes = {k: len(v) for k, v in contrast_values.items()}
n_combos = prod(sizes.values())
print("Group sizes:", sizes)
print("Combinations per essay:", n_combos)
print("Total rows if all essays:", n_combos * df_rq2["essay_id"].nunique())

In [ ]:
# Save the full RQ2 dataframe to Google Drive as a CSV (index=False leaves out the row numbers)
df_rq2.to_csv('/content/drive/MyDrive/df_rq2.csv', index=False)

In [ ]:
# Draw a stratified sample of essays that keeps the same CEFR-score proportions as the full dataset.
# full_counts holds how many essays exist per CEFR level. The function works out how many to take
# from each level so the sample mirrors those proportions (target_n essays in total). It rounds the
# ideal counts down, then hands out any leftover slots to the levels with the largest fractional
# remainders, so the totals add up exactly. Finally it samples that many from each level, shuffles
# the result, and returns it. random_state keeps the sample reproducible.

full_counts = {
    "B2": 187,
    "B1": 157,
    "B1/B2": 136,
    "A2/B1": 77,
    "B2/C1": 56,
    "C1": 19,
    "A2": 11,
}

def ce_fr_stratified_sample(df, target_n=35, counts=full_counts, strata_col="CEFRscore", random_state=42):
    # Work out the ideal (fractional) number to take from each CEFR level
    total = sum(counts.values())
    ideal = {label: (cnt / total) * target_n for label, cnt in counts.items()}
    # Round each down to a whole number, then track the leftover fractions
    base = {label: int(np.floor(val)) for label, val in ideal.items()}
    remainders = {label: ideal[label] - base[label] for label in ideal}
    # Hand out the remaining slots to the levels with the biggest leftover fractions
    remaining = target_n - sum(base.values())
    for label in sorted(remainders, key=remainders.get, reverse=True)[:remaining]:
        base[label] += 1

    # Sample the chosen number of essays from each CEFR level
    rng = np.random.default_rng(random_state)
    parts = []
    for label, take in base.items():
        if take == 0:
            continue
        bucket = df[df[strata_col] == label]
        if len(bucket) < take:
            raise ValueError(f"Not enough rows to sample {take} rows for {label}")
        parts.append(bucket.sample(n=take, random_state=int(rng.integers(1_000_000_000))))

    # Combine all levels, shuffle, and reset the index
    return pd.concat(parts).sample(frac=1, random_state=random_state).reset_index(drop=True)

In [ ]:
# Load the previously saved sample back in from Google Drive
sample_df = pd.read_csv('/content/drive/MyDrive/sample_df.csv')

In [ ]:
# I want to upload the df_rq1 dataframe we made from my google drive
df_rq1 = pd.read_csv('/content/drive/MyDrive/df_rq1.csv')

In [ ]:
df_rq2 = pd.read_csv('/content/drive/MyDrive/df_rq2.csv')

# Before we proceed with Experiment 1. We need to download the necessities to run LLMs

In [ ]:
# Installing hugging face CLI to load models from
!pip install huggingface_hub

In [ ]:
# Logging into huggingFace, token: xxxxxxxxxxxxxxxxxxxxxxx
import huggingface_hub
huggingface_hub.login()

In [ ]:
# Installing transformers to load models, accelerate for model placement, bitsandbytes for quantization and sentence piece which is the tokenizer for llama
!pip install transformers accelerate bitsandbytes sentencepiece


In [ ]:
# opening/importing df_rq2 from myDrive
df_rq2 = pd.read_csv('/content/drive/MyDrive/df_rq2.csv')

In [ ]:
# Load the Llama 3.1 8B Instruct model as a text-generation pipeline.
# bfloat16 keeps memory use down, and device_map="auto" spreads the model across
# available hardware (e.g. the GPU). The pad token is set to the end-of-sequence
# token so the pipeline can process essays in batches, and 'terminators' lists the
# token IDs that tell generation when to stop.

import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "meta-llama/Llama-3.1-8B-Instruct" #Using the modelname

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"dtype": torch.bfloat16},
    device_map="auto",
)

# Fix for ValueError: Pipeline with tokenizer without pad_token cannot do batching.
pipeline.tokenizer.pad_token = pipeline.tokenizer.eos_token

# For stopping at end-of-turn / end-of-sequence
terminators = [
    pipeline.tokenizer.eos_token_id,
    pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

In [ ]:
'''
So, based on Malik et al., we are going to use three different approaches for the prompts. The first
approach mentioned is simply baseline, this directly asks the LLM to generate a certain CEFR grading based on its existing knowledge to guide generation.
'''

# Build the chat messages for the baseline prompt.
# Creates a system message (in Norwegian) telling the model the rules: pick exactly one
# CEFR level, give no explanation, and answer in a fixed format. The user message then
# supplies the task text and the student's essay and asks which CEFR level fits best.

def build_messages_base(text, prompt_task):
    system_content = f"""
Du skal svare på et spørsmål om CEFR-vurdering av en norsk andrespråkstekst.

Regler:
- Velg kun ett CEFR-nivå fra svaralternativene.
- Ikke skriv noen forklaring.
- Ikke bruk andre karaktersystemer enn CEFR-nivåene A2, B1, B2, C1.

Svarformat:
CEFR: <A2|B1|B2|C1>
""".strip()

    user_content = f"""

Oppgavetekst:
{prompt_task}

Studenttekst:
{text}

Spørsmål:
Hvilket CEFR-nivå passer best for denne teksten?

Svaralternativer:
A2
B1
B2
C1

Svar:
""".strip()

    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
    ]

# Run the baseline prediction for one essay.
# Builds the messages, feeds them to the model (greedy/deterministic since do_sample=False),
# pulls the generated reply text out of the output, then uses a regex to find the first
# CEFR level (A2/B1/B2/C1) in that reply. Returns both the full reply and the extracted label
# (or None if no level was found).

def predict_cefr_llama_base(prompt_task, text):
    messages = build_messages_base(text, prompt_task)

    outputs = pipeline(
        messages, # essay and prompt task
        max_new_tokens=256,
        eos_token_id=terminators,
        do_sample=False
    )

    conv = outputs[0]["generated_text"]

    if isinstance(conv, list) and len(conv) > 0 and isinstance(conv[-1], dict):
        full = conv[-1].get("content", "")
    else:
        #Fallback to a string, if there is one
        full = str(conv)

    m = re.search(r"\b(A2|B1|B2|C1)\b", full)
    label = m.group(1) if m else None
    return full, label

In [ ]:
# I want to check whether the context window is long enough so that the models actually read the whole prompt
context_window = pipeline.model.config.max_position_embeddings
print(f"Max context window size: {context_window}")



In [ ]:
# Quick test of the baseline predictor on a single essay (the first row of df_rq1).
# Prints the model's raw answer alongside the true CEFR score so you can eyeball
# whether it's working before running the whole dataset.

example = df_rq1.iloc[0]

full_output, predicted_label = predict_cefr_llama_base(
    prompt_task=example["prompt_task"],
    text=example["text_original"],
)

print("Llama 3 Answer:")
print(repr(full_output))
print("\nGold label:", example["CEFRscore"])

## Stability test for Experiment 1

In [ ]:
'''
Quick test of the baseline predictor on a single essay (the first row of df_rq1).
Prints the model's raw answer alongside the true CEFR score so you can eyeball
whether it's working before running the whole dataset.
'''

example = df_rq1.iloc[0]

full_output, predicted_label = predict_cefr_llama_base(
    prompt_task=example["prompt_task"],
    text=example["text_original"],
)

print("Llama 3 Answer:")
print(repr(full_output))
print("\nGold label:", example["CEFRscore"])

In [ ]:
'''
For our output stability test, I'm basing it on the paper by Atil et al. (2024) where they investigate LLM variation of inputs despite deterministic hyper-parameters and same input.
In their study, they do 5 test runs, they find that stability varies on the task and that LLMs are rarely 100% stable across the same five runs. I'll do 10 test runs though for safety
'''

'''
Because we want to use QWK, I need to use a dictionary to map CEFR labels to numeric indexes
'''

cefr_grades = ["A2", "A2/B1", "B1", "B1/B2", "B2", "B2/C1", "C1"] # these are the labels present in my dataset
cefr_to_index = {label: index for index, label in enumerate(cefr_grades)}

def output_stability_test(sample_df,predict_fn, n_runs = 10, model_name ='model', label_map =cefr_to_index):
  """Test how stable the model's predictions are when given the same essay multiple times.

  For each essay, run the predictor n_runs times (default 10) and collect the labels.
  Then record the majority label, how many unique answers came up, whether it was fully
  stable (same answer every time), and what fraction of runs the majority answer got.
  Labels are also converted to ordinal indices (for QWK later), and a flag marks any
  essay where a gold or predicted label wasn't found in the mapping.
  """
  rows = []
  for index, row in sample_df.iterrows():
    # Run the same essay through the model n_runs times and collect each prediction
    preds = []
    for _ in range(n_runs):
      _, label = predict_fn(
          prompt_task=row["prompt_task"],
          text=row["text_original"],
      )
      preds.append(label)
    # Find the most common prediction across the runs
    counts = Counter(preds)
    majority_label, majority_count = counts.most_common(1)[0]
    gold_label = row['CEFRscore']
    # We need to map the labels to ordinal indices
    gold_index = label_map.get(gold_label)
    pred_indices = [label_map.get(pred) for pred in preds]
    majority_index = label_map.get(majority_label)
    rows.append({
          'row_index': index,
          'gold_label': row['CEFRscore'],
          'gold_index': gold_index,
          'prediction_indices': pred_indices,
          'predictions': preds,
          'unique_predictions': len(counts),
          'stable': len(counts)== 1,
          'majority_label': majority_label,
          'majority_index': majority_index,
          'majority_fraction': majority_count / n_runs,
          'has_unmapped_label': (gold_index is None) or any(pred_index is None for pred_index in pred_indices),
    })
  return pd.DataFrame(rows)

In [ ]:
stability_df_llama = output_stability_test(sample_df, predict_cefr_llama_base, n_runs = 10, model_name = 'llama', label_map = cefr_to_index )


In [ ]:
'''
I want to make a function that prints out the stability percentage and also count of stable rows
'''

def stability_summary(
    stability_df,
    model="name",
    stable_col="stable",
    label_col="gold_label",
    idx_col="gold_index",
    label_map=cefr_to_index,
):
    """Print a short summary of the stability results.

    Reports the percentage of essays that were fully stable and the raw count out of the
    total. Also lists which CEFR labels appeared in the data (with their numeric indices)
    and which indices were seen, ordered by the CEFR scale.
    """

    stable_series = stability_df[stable_col].astype(bool)
    stable_pct = stable_series.mean() * 100
    stable_count = stable_series.sum()
    stable_total = len(stability_df)

    print(f"{model} - Percentage of stable outputs: {stable_pct:.2f}%")
    print(f"{model} - Stable rows: {stable_count}/{stable_total}")

    seen_labels = stability_df[label_col].dropna().unique().tolist()
    seen_indices = sorted(set(stability_df[idx_col].dropna().astype(int)))

    ordered_seen = [
        label for label in cefr_grades if label in seen_labels
    ]
    mapping = ", ".join(f"{label}({label_map[label]})" for label in ordered_seen)
    print(f"{model} - Labels seen: {mapping or 'none'}")
    print(f"{model} - Numeric indices seen: {', '.join(str(idx) for idx in seen_indices)}")

In [ ]:
stability_summary(stability_df_llama, model='llama3')

In [ ]:
from sklearn.metrics import cohen_kappa_score

def compute_qwk(df, gold_col = 'gold_index', pred_col = 'prediction_index', label_map = cefr_to_index, weights ='quadratic'):
    """Compute the quadratic weighted kappa (QWK) between gold and predicted CEFR indices.

    Drops rows with unmapped labels first (warning if any), errors out if nulls remain,
    then returns a small report with the score and how many essays were kept or dropped.

    Args:
        df: One row per essay, with the index columns and a 'has_unmapped_label' flag.
        gold_col: Column holding the true CEFR index.
        pred_col: Column holding the predicted CEFR index.
        label_map: CEFR-label-to-index mapping; its values set the full label scale.
        weights: Weighting scheme for cohen_kappa_score (quadratic suits ordinal levels).
    """

    total = len(df)
    # if there are any empty values, null values, I would like to instead filter to only rows where both labels are mapped
    labeled_df = df[df['has_unmapped_label'] == False].copy()
    dropped_rows = len(df) - len(labeled_df)

    if dropped_rows > 0:
      print(f"Warning: Dropping {dropped_rows}/{len(df)} rows with unmapped labels before computing QWK")

    if labeled_df[[gold_col, pred_col]].isnull().any().any():
      raise ValueError("Gold and prediction columns contain null values. Check mapping first")

    qwk = cohen_kappa_score(
      labeled_df[gold_col],
      labeled_df[pred_col],
      labels=list(label_map.values()),
      weights=weights,
  )

    report = {
      'qwk': qwk,
      'evaluated': len(labeled_df),
      'dropped essays': dropped_rows,
      'dropout_rate': round(dropped_rows / total, 4),
  }

    return report

In [ ]:
'''
To collect all predictions for each model, I am going to create a function that takes in a couple of parameters, including the predict cefr function
'''

def collect_cefr_predictions(df, predict_fn, label_map=cefr_to_index, model_name="model"):
    """Run a prediction function over every essay and collect the results into a dataframe.

    For each essay it stores the gold and predicted labels, their numeric indices, the raw
    model output (kept so ungraded entries can be inspected), and a flag for unmapped labels.

    Args:
        df: One row per essay, with 'CEFRscore', 'prompt_task', and 'text_original' columns.
        predict_fn: A predictor returning (full_output, predicted_label) for one essay.
        label_map: CEFR-label-to-index mapping used for the gold and prediction indices.
        model_name: Label stored in the 'model' column to identify which model produced the row.
    """

# Initiating a list to hold all of my outputs
  rows = []

  for index, row in df.iterrows():
    gold_label = row['CEFRscore']
    full_output, pred_label = predict_fn(
        prompt_task = row['prompt_task'],
        text = row['text_original'],
    )

    rows.append({
        'model': model_name,
        'row_index': index,
        'gold_label': gold_label,
        'gold_index':label_map.get(gold_label),
        'raw_output': full_output, #saving the model output, to see in case some entries are not graded
        'prediction_label': pred_label,
        'prediction_index': label_map.get(pred_label),
        'has_unmapped_label': (gold_label not in label_map) or (pred_label not in label_map),
    })

  return pd.DataFrame(rows)

# Part 3.1 - Using LLAMA to grade student essays

In [ ]:
'''
Run the baseline (zero-shot) Llama inference over the full RQ1 dataset.
Collects every prediction into pred_llama_rq1_base, tagged as model "llama3",
ready to save as a CSV afterwards.
'''
pred_llama_rq1_base = collect_cefr_predictions(df_rq1, predict_cefr_llama_base, model_name="llama3")

In [ ]:
# I want to save this as a csv file
pred_llama_rq1_base.to_csv('pred_llama_rq1_base.csv', index=False)

In [ ]:
# After completion, I want to print out how many rows there are and how many are missing a prediction
print(f"Total rows: {len(pred_llama_rq1_base)}")
print(f"Missing predictions: {pred_llama_rq1_base['prediction_label'].isna().sum()}")

In [ ]:
pred_llama_rq1_base.head() #just checking the first/last 5 rows using either head() or tail()

In [ ]:
# Print the QWK score for the baseline Llama predictions, read from the report dict
print(f"QWK (majority prediction): {compute_qwk(pred_llama_rq1_base)['qwk']:.4f}")

In [ ]:
cefr_grades = ['A2', 'B1', 'B2', 'C1']

cefr_ranks = {label: i for i, label in enumerate(cefr_grades)}

def cefr_set_evaluation(df, gold_col='gold_label', pred_col='prediction_label', classes= cefr_grades, inplace=False, zero_division=0):
  """Score CEFR predictions while treating hybrid gold labels (e.g. 'B1/B2') as a set of acceptable answers.

  A prediction counts as correct if it falls inside the gold set, so 'B1' is accepted when
  the gold is 'B1/B2'. Reports an overall set-based accuracy plus a per-class precision/recall/
  F1 report (with macro and weighted averages), scoring only rows that have both a gold and a
  prediction.

  Args:
      df: One row per essay, with the gold and prediction label columns.
      gold_col: Column holding the true CEFR label (may be hybrid like 'B1/B2').
      pred_col: Column holding the predicted CEFR label.
      classes: The CEFR levels to report on.
      inplace: If True, add working columns to df directly instead of a copy.
      zero_division: Value used for precision/recall/F1 when the denominator is zero.
  """

  def normalize_labels(label):
    '''
    This first function is to remove whitespace and make all labels uppercase
    because this is how they're represented in the dataframe.
    '''
    if pd.isna(label):
      return pd.NA
    return str(label).strip().upper()


  def gold_set(gold_label):
    '''
    This function is to ensure that hybrid-answers in the gold_label column like 'B1/B2'
    gets turned into {'B1', 'B2'} so that an answer that's either 'B1' or 'B2' gets accepted.
    '''
    gold_label = normalize_labels(gold_label)

    if pd.isna(gold_label):
      return pd.NA

    # Set comprehension syntax: {expression for item in iterable if condition}
    return {x.strip() for x in gold_label.split('/') if x.strip()}


  def pred_sets_accept(gold_label, prediction_label):
    '''
    This function is to ensure that the prediction is acceptable if and only if it is in the gold set.
    '''
    gold_n = normalize_labels(gold_label)
    prediction_n = normalize_labels(prediction_label)

    if pd.isna(gold_n) or pd.isna(prediction_n):
      return pd.NA

    return prediction_n in gold_set(gold_n)


  '''
  Here I want to build the two columns to hold the answer to the dataframes
  '''

  dataframe = df if inplace else df.copy()



  dataframe['y_true'] = dataframe[gold_col].apply(gold_set) # Applying the function to ensure that hybrid gold answers get turned into a set
  dataframe['y_pred'] = dataframe[pred_col].apply(normalize_labels) # Applying the function to normalise the labels

  '''
  I want to keep the rows where the model produces a prediction, these are kept and a new dataframe is made from them.
  '''

  boolean_mask = dataframe['y_pred'].notna() & dataframe['y_true'].notna()
  scored = dataframe.loc[boolean_mask].copy()

  scored['is_correct_pred'] = [
      (pred in golds) for pred, golds in zip(scored['y_pred'], scored['y_true'])
      ]


  membership_accuracy = float(np.mean(scored['is_correct_pred'])) if len(scored) else np.nan

  n_total_rows = len(dataframe)
  n_scored = len(scored)

  print(f"Set-based accuracy: {membership_accuracy:.4f}")
  print(f"Scored {n_scored}/{n_total_rows} rows ({n_scored/n_total_rows:.1%})")

  '''
  We have now calculated the membership (set) accuracy based on Chzhen et al. (2021). In this section, I'm going to do the set-aware per-class report based on Zhang & Zhou (2024)
  '''

  rows = []

  support = {c:0 for c in classes} #support here refers to how many instances accept class c, i.e., is c in the gold set?

  for golds in scored['y_true']:
    for c in classes:
      if c in golds:
        support[c] += 1



  for c in classes:
    tp = fp = fn = 0

    for golds, pred in zip(scored['y_true'], scored['y_pred']):
      in_golds = (c in golds)
      pred_is_c = (pred == c)

      if pred_is_c and in_golds:
        tp += 1
      elif pred_is_c and not in_golds:
        fp += 1
      elif (not pred_is_c) and in_golds:
        fn += 1

    precision = tp / (tp + fp) if (tp + fp) else float(zero_division)
    recall = tp / (tp + fn) if (tp + fn) else float(zero_division)
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else float(zero_division)

    rows.append({
      'label': c,
      'precision': precision,
      'recall': recall,
      'f1-score': f1,
      'support' : support[c],
      'tp': tp,
      'fp': fp,
      'fn': fn,
      })

  report = pd.DataFrame(rows).set_index('label')

  '''
  Here I want to calculate the macro average to treat each class equally
  '''

  macro_avg = report[['precision', 'recall', 'f1-score']].mean()

  '''
  Here I want to calculate the weighted average
  '''
  total_support = report['support'].sum()
  if total_support > 0:
    weights = report['support'] / total_support
    weighted = (report[['precision', 'recall', 'f1-score']].mul(weights, axis=0)).sum()
  else:
    weighted = pd.Series({'precision': np.nan, 'recall': np.nan, 'f1-score': np.nan})


  report_summary = report[['precision', 'recall', 'f1-score', 'support']].copy()
  report_summary.loc['macro avg'] = {
      'precision':macro_avg['precision'],
      'recall': macro_avg['recall'],
      'f1-score': macro_avg['f1-score'],
      'support': total_support,
  }

  report_summary.loc['weighted avg'] = {
      'precision': weighted['precision'],
      'recall': weighted['recall'],
      'f1-score': weighted['f1-score'],
      'support': total_support,
  }

  return scored, membership_accuracy, report_summary

In [ ]:
def count_correct_predictions(df, correct_col='is_correct_pred'):
  """Count how many predictions were correct versus incorrect.

  Args:
      df: A scored dataframe containing the correctness flag column.
      correct_col: Boolean column marking whether each prediction was correct.
  """
  return df[correct_col].value_counts(dropna=False)

In [ ]:
# Show the correct vs incorrect prediction counts for the baseline Llama run
count_correct_predictions(scored_llama_df_base)

In [ ]:
def pred_label_distribution(df, pred_col='y_pred', drop_na=True):
  """Show the distribution of predicted CEFR labels as proportions.

  Args:
      df: A dataframe containing the prediction column.
      pred_col: Column holding the predicted labels.
      drop_na: If True, leave missing predictions out of the proportions.
  """
  return df[pred_col].value_counts(normalize=True, dropna=drop_na)

In [ ]:
# Show the predicted CEFR label distribution (as proportions) for the baseline Llama run
pred_label_distribution(scored_llama_df_base, pred_col="y_pred")

In [ ]:
def gold_label_distribution(df, gold_col='y_true', drop_na=True):
  """Show the distribution of gold CEFR labels as proportions.

  Args:
      df: A dataframe containing the gold label column.
      gold_col: Column holding the gold labels.
      drop_na: If True, leave missing labels out of the proportions.
  """
  return df[gold_col].value_counts(normalize=True, dropna=drop_na)

In [ ]:
# Show the distribution of gold CEFR label sets (hybrids like {B1, B2} counted as their own
# category) as proportions, for the baseline Llama run
gold_label_distribution(scored_llama_df_base)

In [ ]:
def classification_report_df(
    df,
    gold_col='gold_label',
    pred_col='prediction_label',
    classes=cefr_grades,
    inplace=False,
    zero_division=0
):
    """
    Returns the manual, set-aware classification report we built with precision/recall/F1 per class
    produced by cefr_set_evaluation().
    """
    _, _, report_summary = cefr_set_evaluation(
        df,
        gold_col=gold_col,
        pred_col=pred_col,
        classes=classes,
        inplace=inplace,
        zero_division=zero_division
    )
    return report_summary


In [ ]:
# Build and print the set-aware classification report (precision/recall/F1 per class) for the baseline Llama run
report_llama_base = classification_report_df(
    pred_llama_rq1_base,
    gold_col="gold_label",
    pred_col="prediction_label"
)

print(report_llama_base)

In [ ]:
'''
Moving on, I'm proceeding to the next approach for improving performance. This entails including concrete descriptions of CEFR levels in the prompt.
Malik et al. proposes that you can describe either just the target level or every single level. We use descriptions from HKDIR.
'''

def build_messages_cefr_desc(text, prompt_task):
    """Build the chat messages for the level-description prompt.

    Same structure as the baseline prompt, but the system message now includes the full
    HKDIR descriptions of each CEFR level (A2-C1) to give the model explicit grading criteria.
    """
    system_content = f"""
Du skal svare på et spørsmål om CEFR-vurdering av en norsk andrespråkstekst.


Her vurderingskriterier:

Skalaene beskriver hva innlærere kan gjøre. Fokuset er altså på hva man kan, ikke feil og mangler. Her er global skala, som er en kortfattet oppsummering av nivåene:

Evaluering: [C1]
Beskrivelse: Kan forstå et bredt spekter av lengre, krevende tekster og oppfatte budskap som ikke er direkte uttrykt. Kan uttrykke seg flytende og spontant uten at det merkes noe særlig at en leter etter uttrykksmåter. Kan bruke språket fleksibelt og hensiktsmessig til sosiale, akademiske og yrkesrelaterte formål. Kan produsere klare, velstrukturerte og detaljerte tekster om komplekse emner og vise at en mestrer ulike setningsmønstre, bindeledd og sammenbindende markører.

Evaluering: [B2]
Beskrivelse: Kan forstå hovedinnholdet i komplekse tekster om både konkrete og abstrakte emner, også faglige drøftinger innenfor ens eget fagområde. Kan delta i samtaler med et så spontant og flytende språk at regelmessig kommunikasjon med brukere av målspråket ikke blir anstrengende for noen av partene. Kan produsere klare, detaljerte tekster om et vidt spekter av emner, og forklare et synspunkt på en aktuell sak og gi argumenter for og imot ulike alternativer.

Evaluering: [B1]
Beskrivelse: Kan forstå hovedpunktene i klar, standard input om kjente emner som en ofte møter i forbindelse med arbeid, skole, fritid osv. Kan klare seg i de fleste situasjoner som kan oppstå når en reiser i et område der språket snakkes. Kan produsere enkle, sammenhengende tekster om emner som er kjente eller av personlig interesse. Kan beskrive opplevelser og hendelser, drømmer, håp og planer, og kort forklare og begrunne meninger og planer.

Evaluering: [A2]
Beskrivelse: Kan forstå setninger og vanlige uttrykk knyttet til de viktigste områdene av dagliglivet (f.eks. svært enkel informasjon om en selv og familien, innkjøp, nærmiljø og arbeidsliv). Kan klare seg i enkle og rutinepregede samtalesituasjoner med direkte utveksling av informasjon om kjente og rutinepregede forhold. Kan med enkle ord/tegn beskrive visse sider ved sin egen bakgrunn og sitt nærmiljø og grunnleggende personlige behov.

Regler:
- Velg kun ett CEFR-nivå fra svaralternativene.
- Ikke skriv noen forklaring.
- Ikke bruk andre karaktersystemer enn CEFR-nivåene A2, B1, B2, C1.

Svarformat:
CEFR: <A2|B1|B2|C1>
""".strip()

    user_content = f"""

Oppgavetekst:
{prompt_task}

Studenttekst:
{text}

Spørsmål:
Hvilket CEFR-nivå passer best for denne teksten?

Svaralternativer:
A2
B1
B2
C1

Svar:
""".strip()

    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
    ]



def predict_cefr_desc(prompt_task, text):
    """Run the level-description prediction for one essay.

    Builds the description-rich messages, runs deterministic generation, pulls the reply text
    out of the output, and uses a regex to extract the CEFR level. Returns the full reply and
    the extracted label (or None if none was found).
    """
    messages = build_messages_cefr_desc(text, prompt_task)

    outputs = pipeline(
        messages,
        max_new_tokens=256,
        eos_token_id=terminators,
        do_sample=False,
        temperature=0.0,
    )

    conv = outputs[0]["generated_text"]

    # For Llama 3 chats: conv is a list of {"role": ..., "content": ...}
    if isinstance(conv, list) and len(conv) > 0 and isinstance(conv[-1], dict):
        full = conv[-1].get("content", "")
    else:
        # Fallback to a string, if we have one
        full = str(conv)

    m = re.search(r"\b(A1|A2|B1|B2|C1|C2)\b", full)
    label = m.group(1) if m else None
    return full, label

In [ ]:
# Run the level-description Llama inference over the full RQ1 dataset.
# Collects every prediction into pred_llama_rq1_cefr_desc, tagged as model
# "LLama3 - CEFR descriptions", ready to score afterwards.
pred_llama_rq1_cefr_desc = collect_cefr_predictions(df_rq1, predict_cefr_desc, model_name='LLama3 - CEFR descriptions')

In [ ]:
# After completion, i want to print out how many rows there are and how many that are missing a prediction
print(f"Total rows: {len(pred_llama_rq1_cefr_desc)}")
print(f"Missing predictions: {pred_llama_rq1_cefr_desc['prediction_label'].isna().sum()}")

In [ ]:
# Print the QWK score for the CEFR-description Llama predictions, read from the report dict
print(f"QWK (majority prediction): {compute_qwk(pred_llama_rq1_cefr_desc)['qwk']:.4f}")

In [ ]:
# Show the predicted CEFR label distribution (as proportions) for the CEFR-description Llama run
pred_label_distribution(scored_llama_df_cefr_desc, pred_col="y_pred")

In [ ]:
# Build and print the set-aware classification report (precision/recall/F1 per class) for the CEFR-description Llama run
report_llama_cefr_desc = classification_report_df(
    pred_llama_rq1_cefr_desc,
    gold_col="gold_label",
    pred_col="prediction_label"
)

print(report_llama_cefr_desc)

In [ ]:
'''
This is the final prompting strategy from Malik et al., the same paper the previous approaches referenced.
It builds on the level-description prompt by also adding one worked example essay per CEFR level (a few-shot setup).
'''


def build_messages_one_shot(text, prompt_task):
    """Build the chat messages for the one-shot prompt.

    Extends the level-description prompt by including one example learner essay per CEFR
    level (A2-C1), so the model has one concrete reference per level alongside the written
    criteria before grading.
    """
    system_content = f"""
Du skal svare på et spørsmål om CEFR-vurdering av en norsk andrespråkstekst.

Her vurderingskriterier:

Skalaene beskriver hva innlærere kan gjøre. Fokuset er altså på hva man kan, ikke feil og mangler. Her er global skala, som er en kortfattet oppsummering av nivåene:

Evaluering: [C1]
Beskrivelse: Kan forstå et bredt spekter av lengre, krevende tekster og oppfatte budskap som ikke er direkte uttrykt. Kan uttrykke seg flytende og spontant uten at det merkes noe særlig at en leter etter uttrykksmåter. Kan bruke språket fleksibelt og hensiktsmessig til sosiale, akademiske og yrkesrelaterte formål. Kan produsere klare, velstrukturerte og detaljerte tekster om komplekse emner og vise at en mestrer ulike setningsmønstre, bindeledd og sammenbindende markører.

Evaluering: [B2]
Beskrivelse: Kan forstå hovedinnholdet i komplekse tekster om både konkrete og abstrakte emner, også faglige drøftinger innenfor ens eget fagområde. Kan delta i samtaler med et så spontant og flytende språk at regelmessig kommunikasjon med brukere av målspråket ikke blir anstrengende for noen av partene. Kan produsere klare, detaljerte tekster om et vidt spekter av emner, og forklare et synspunkt på en aktuell sak og gi argumenter for og imot ulike alternativer.

Evaluering: [B1]
Beskrivelse: Kan forstå hovedpunktene i klar, standard input om kjente emner som en ofte møter i forbindelse med arbeid, skole, fritid osv. Kan klare seg i de fleste situasjoner som kan oppstå når en reiser i et område der språket snakkes. Kan produsere enkle, sammenhengende tekster om emner som er kjente eller av personlig interesse. Kan beskrive opplevelser og hendelser, drømmer, håp og planer, og kort forklare og begrunne meninger og planer.

Evaluering: [A2]
Beskrivelse: Kan forstå setninger og vanlige uttrykk knyttet til de viktigste områdene av dagliglivet (f.eks. svært enkel informasjon om en selv og familien, innkjøp, nærmiljø og arbeidsliv). Kan klare seg i enkle og rutinepregede samtalesituasjoner med direkte utveksling av informasjon om kjente og rutinepregede forhold. Kan med enkle ord/tegn beskrive visse sider ved sin egen bakgrunn og sitt nærmiljø og grunnleggende personlige behov.

EKSEMPLER PÅ NIVÅER:

[A2-eksempler]
'''
Når man gifter seg i hjemlandet mitt,må man gjøre mange ting. De feirer og lager asiatisk mat. De tar
på seg fint kjolen og dress.Naboene og venner kommer til flest. De drikker øl. Mann gifter seg om morgen.
Mannen må gi foreldre til dame med penger. Dama må bo sammen med mannen.
Da jeg gifter meg, bodde jeg sammen med familien min i Asia. Fordi min mann har jobb i Norge. Jeg har
flytte hjem til Norge for1 år siden.
'''

[B1-eksempler]
'''
I hjemlandet mitt det er vanlig for man å gifte seg når man er 26-27 år gammel. Men det er også
viktig for man å gifte seg når man har jobb og egen hus. For eksempel, jeg kan fortelle om hvordan min
bror gifte seg for 2 år siden. Omtrent 4 år siden han var forelske med ei jente som heter Anna. Hun var
også forelske med han. De brukte langt tid å kjenne hverandre. Forresten det er ikke vanlig å ha sam-
boer i hjemlandet mitt. Etter en stund de bestemte deres å ha byryllup. Det var en stor byryllupfest og
200 mennesker var invitertet. Det var fantastik byryllup,alle var fornøyd og hadde gøy. Det var mange
forskjellige mat of drikke. Vi danset mye og spilte forskjellige spiller. Byryllupen startet kolkka 6 i kveld og
sluttet klokka 12 midnatt.
'''

[B2-eksempler]
'''
Jeg synes det er en kjempe god ide hvis ansatte blir gitt et visst antall fridager de kan bruke når de vil. Det
betyr at de som er religiøse - ikke bare kristne - kan ta fri i sine religiøse høytider, og de som er ikke religi-
øse som meg kan planlegge våre egne fridager bedre! Spør dere meg, er det en vinn-vinn for alle!
Samtidig, jeg lurer litt på hva slags påvirkning dette kan ha på skoledager, for eksempel. Når skulle læ-
rerne ta fri? Skoleferie per idag er selvfølgelig knyttet til kristne høytider. Hvis flere lærere vil heller jobbe
i jule- eller påsketid, vil vi få kaos, tror jeg! Det kan ikke løses så enkelt, og vi trenger en nasjonal debatt
rundt temaet.
'''

[C1-eksempler]
'''
To personer, Ola og Kari, snakker om lekser på skolen. Begge forstår at det ofte er veldig slitsomt for barn, men
samtidig er de uenige om hvor viktig det er å ha lekser.
Kari klager på at datteren hennes ikke har lyst til å gjøre lekser etter skolen. Hun mener at barn må ha
mulighet til å bare nyte livet og fint vær.
Ola prøver å forklare at lekser er en nødvendig del av undervisningen, fordi de forbereder barn til framtiden.
Ved å gjøre lekser, lærer barn å løse problemer selv og fokusere på ting som de ikke vil gjøre. Han legger til at
voksne også trenger å jobbe, selv når de ikke vil det, og at det er fint at barn får trene litt før de selv blir voksne.
Kari er uenig med denne påstanden. Hun synes at barn må være seg selv og trenger ikke å ha samme problemer
som voksne for tidlig. Hun sier også at voksne kan velge hvor og når de jobber, mens barn er tvunget til å gå
på skole.
'''

Regler:
- Velg kun ett CEFR-nivå fra svaralternativene.
- Ikke skriv noen forklaring.
- Ikke bruk andre karaktersystemer enn CEFR-nivåene A2, B1, B2, C1.

Svarformat:
CEFR: <A2|B1|B2|C1>
""".strip()

    user_content = f"""

Oppgavetekst:
{prompt_task}

Studenttekst:
{text}

Spørsmål:
Hvilket CEFR-nivå passer best for denne teksten?

Svaralternativer:
A2
B1
B2
C1

Svar:
""".strip()

    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
    ]


def predict_cefr_example(prompt_task, text):
    """Run the one-shot prediction for one essay.

    Builds the example-rich messages, runs deterministic generation, pulls the reply text
    out of the output, and uses a regex to extract the CEFR level. Returns the full reply and
    the extracted label (or None if none was found).
    """
    messages = build_messages_one_shot(text, prompt_task)

    outputs = pipeline(
        messages,
        max_new_tokens=256,
        eos_token_id=terminators,
        do_sample=False,
    )

    conv = outputs[0]["generated_text"]

    # For Llama 3 chats: conv is a list of {"role": ..., "content": ...}
    if isinstance(conv, list) and len(conv) > 0 and isinstance(conv[-1], dict):
        full = conv[-1].get("content", "")
    else:
        # Fallback to a string, if we have one
        full = str(conv)

    m = re.search(r"\b(A1|A2|B1|B2|C1|C2)\b", full)
    label = m.group(1) if m else None
    return full, label

In [ ]:
# Run the one-shot Llama inference over the full RQ1 dataset.
# Collects every prediction into pred_llama_rq1_one_shot, tagged as model
# "LLama3 - One shot", ready to score afterwards.
pred_llama_rq1_one_shot = collect_cefr_predictions(df_rq1, predict_cefr_example, model_name='LLama3 - One shot')

In [ ]:
# Saving this as a csv file
pred_llama_rq1_one_shot.to_csv('pred_llama_rq1_few_shot.csv', index=False)

In [ ]:
# After completion, i want to print out how many rows there are and how many that are missing a prediction
print(f"Total rows: {len(pred_llama_rq1_one_shot)}")
print(f"Missing predictions: {pred_llama_rq1_one_shot['prediction_label'].isna().sum()}")

In [ ]:
# Print the QWK score for the one-shot Llama predictions, read from the report dict
print(f"QWK (majority prediction): {compute_qwk(pred_llama_rq1_one_shot)['qwk']:.4f}")

In [ ]:
# Show the predicted CEFR label distribution (as proportions) for the one-shot Llama run
pred_label_distribution(scored_llama_df_one_shot, pred_col="y_pred")

In [ ]:
# Build and print the set-aware classification report (precision/recall/F1 per class) for the one-shot Llama run
report_llama_one_shot = classification_report_df(
    pred_llama_rq1_one_shot,
    gold_col="gold_label",
    pred_col="prediction_label"
)

print(report_llama_one_shot)

# Part 3.2 - Using NorMistral-7b-warm-instruct to grade student essays

In [ ]:
# Load the NorMistral-7B-warm-instruct model, a Norwegian instruction-tuned model used here
# as a second model to compare against Llama. The tokenizer turns text into tokens, the model
# is loaded in bfloat16 to save memory and spread across available hardware (device_map='auto'),
# and model.eval() switches it to inference mode (no training behaviour like dropout).
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = 'norallm/normistral-7b-warm-instruct'

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map = 'auto',
    torch_dtype = torch.bfloat16
)

model.eval()

In [ ]:
# Print NorMistral's maximum context window (how many tokens it can take at once)
print("NorMistral context window:", pipeline.model.config.max_position_embeddings)

In [ ]:
def build_messages_base(text, prompt_task):
    """Build the chat messages for the baseline prompt (NorMistral version).

    Same baseline structure as the Llama version: a system message with the rules and a user
    message supplying the task and student text. Redefined here for the NorMistral runs.
    """
    system_content = f"""
Du skal svare på et spørsmål om CEFR-vurdering av en norsk andrespråkstekst.

Regler:
- Velg kun ett CEFR-nivå fra svaralternativene.
- Ikke skriv noen forklaring.
- Ikke bruk andre karaktersystemer enn CEFR-nivåene A2, B1, B2, C1.

Svarformat:
CEFR: <A2|B1|B2|C1>
""".strip()

    user_content = f"""

Oppgavetekst:
{prompt_task}

Studenttekst:
{text}

Spørsmål:
Hvilket CEFR-nivå passer best for denne teksten?

Svaralternativer:
A2
B1
B2
C1

Svar:
""".strip()

    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
    ]


@torch.inference_mode() # A decorator that speeds up model inference by disabling gradient calculations, i.e., not track operations for gradient computation during backward pass and disabling view tracking and version counter bumps
def predict_cefr_normistral(prompt_task, text):
    """Run the baseline prediction for one essay using NorMistral.

    Unlike the Llama version (which uses a transformers pipeline), this calls the model
    directly: it formats the messages with the chat template, moves the tokens to the model's
    device, generates deterministically, decodes only the newly generated tokens, and extracts
    the CEFR level with a regex. Returns the full reply and the extracted label (or None).
    """
    messages = build_messages_base(text, prompt_task)

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        return_tensors = 'pt'
    )
    input_ids = {k: v.to(model.device) for k, v in input_ids.items()}


    output_ids = model.generate(
        **input_ids,
        max_new_tokens = 256,
        do_sample = False,
        use_cache = True,
        )

    gen_ids = output_ids[0, input_ids['input_ids'].shape[-1]:]
    full = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    m = re.search(r"\b(A2|B1|B2|C1)\b", full.upper())
    label = m.group(1) if m else None
    return full, label

In [ ]:
# Quick test of the NorMistral baseline predictor on a single essay (row 89 of df_rq1),
# pulling out the task, text, and gold label so the output can be checked by hand
row = df_rq1.iloc[89]

prompt_task = row['prompt_task']
text = row['text_original']
gold = row["CEFRscore"]

full_output, label = predict_cefr_normistral(prompt_task, text)

In [ ]:
# Print the predicted CEFR level next to the gold label for the single test essay
print("\nPredicted CEFR:")
print(label)

print("\nGold CEFR:")
print(gold)

In [ ]:
# Run the output stability test for NorMistral: predict each essay in the sample 10 times
# and record how consistent the predictions are across runs
stability_df_normistral = output_stability_test(sample_df, predict_cefr_normistral, n_runs = 10, model_name = 'NorMistral')

In [ ]:
# Print the stability summary for NorMistral (percentage of stable essays, counts, and labels seen)
stability_summary(stability_df_normistral, model='normistral')

In [ ]:
# Some essays returned None (no parseable label), but the predictions were still stable
# across all 10 runs in the stability test.

# Run the baseline NorMistral inference over the full RQ1 dataset.
# Collects every prediction into pred_normistral_rq1_cefr_base, tagged as model
# "NorMistral", ready to score afterwards.
pred_normistral_rq1_cefr_base = collect_cefr_predictions(df_rq1, predict_cefr_normistral, model_name="NorMistral")

In [ ]:
# Display the full NorMistral baseline predictions dataframe
pred_normistral_rq1_cefr_base

In [ ]:
# Save the NorMistral baseline predictions to a CSV (index=False leaves out the row numbers)
pred_normistral_rq1_cefr_base.to_csv('pred_normistral_rq1_cefr_base.csv', index=False)

In [ ]:
# After running the model on all the rows, I want to print out how many rows there are and how many are missing a prediction for NorMistral
print(f"Total rows: {len(pred_normistral_rq1_cefr_base)}")
print(f"Missing predictions: {pred_normistral_rq1_cefr_base['prediction_label'].isna().sum()}")

In [ ]:
# Compute and print the QWK score for the NorMistral baseline predictions
print(f"QWK (majority prediction): {compute_qwk(pred_normistral_rq1_cefr_base)['qwk']:.4f}")

In [ ]:
# Show the predicted CEFR label distribution (as proportions) for the NorMistral baseline run
pred_label_distribution(scored_normistral_base, pred_col="y_pred") # Interesting — NorMistral spreads its grades quite differently from Llama (on the essays it does grade)

In [ ]:
# Show the gold CEFR label distribution (over gold sets, hybrids counted as their own category)
# for the NorMistral baseline run
gold_label_distribution(scored_normistral_base)

In [ ]:
# Build and print the set-aware classification report (precision/recall/F1 per class) for the NorMistral baseline run
report_normistral_base = classification_report_df(
    pred_normistral_rq1_cefr_base,
    gold_col="gold_label",
    pred_col="prediction_label"
)

print(report_normistral_base)

In [ ]:
def build_messages_cefr_desc_nm(text, prompt_task):
    """Build the chat messages for the level-description prompt (NorMistral version).

    Same as the Llama description prompt: the system message includes the full HKDIR
    descriptions of each CEFR level. The 'nm' suffix marks this as the NorMistral variant.
    """
    system_content = f"""
Du skal svare på et spørsmål om CEFR-vurdering av en norsk andrespråkstekst.


Her vurderingskriterier:

Skalaene beskriver hva innlærere kan gjøre. Fokuset er altså på hva man kan, ikke feil og mangler. Her er global skala, som er en kortfattet oppsummering av nivåene:

Evaluering: [C1]
Beskrivelse: Kan forstå et bredt spekter av lengre, krevende tekster og oppfatte budskap som ikke er direkte uttrykt. Kan uttrykke seg flytende og spontant uten at det merkes noe særlig at en leter etter uttrykksmåter. Kan bruke språket fleksibelt og hensiktsmessig til sosiale, akademiske og yrkesrelaterte formål. Kan produsere klare, velstrukturerte og detaljerte tekster om komplekse emner og vise at en mestrer ulike setningsmønstre, bindeledd og sammenbindende markører.

Evaluering: [B2]
Beskrivelse: Kan forstå hovedinnholdet i komplekse tekster om både konkrete og abstrakte emner, også faglige drøftinger innenfor ens eget fagområde. Kan delta i samtaler med et så spontant og flytende språk at regelmessig kommunikasjon med brukere av målspråket ikke blir anstrengende for noen av partene. Kan produsere klare, detaljerte tekster om et vidt spekter av emner, og forklare et synspunkt på en aktuell sak og gi argumenter for og imot ulike alternativer.

Evaluering: [B1]
Beskrivelse: Kan forstå hovedpunktene i klar, standard input om kjente emner som en ofte møter i forbindelse med arbeid, skole, fritid osv. Kan klare seg i de fleste situasjoner som kan oppstå når en reiser i et område der språket snakkes. Kan produsere enkle, sammenhengende tekster om emner som er kjente eller av personlig interesse. Kan beskrive opplevelser og hendelser, drømmer, håp og planer, og kort forklare og begrunne meninger og planer.

Evaluering: [A2]
Beskrivelse: Kan forstå setninger og vanlige uttrykk knyttet til de viktigste områdene av dagliglivet (f.eks. svært enkel informasjon om en selv og familien, innkjøp, nærmiljø og arbeidsliv). Kan klare seg i enkle og rutinepregede samtalesituasjoner med direkte utveksling av informasjon om kjente og rutinepregede forhold. Kan med enkle ord/tegn beskrive visse sider ved sin egen bakgrunn og sitt nærmiljø og grunnleggende personlige behov.

Regler:
- Velg kun ett CEFR-nivå fra svaralternativene.
- Ikke skriv noen forklaring.
- Ikke bruk andre karaktersystemer enn CEFR-nivåene A2, B1, B2, C1.

Svarformat:
CEFR: <A2|B1|B2|C1>
""".strip()

    user_content = f"""

Oppgavetekst:
{prompt_task}

Studenttekst:
{text}

Spørsmål:
Hvilket CEFR-nivå passer best for denne teksten?

Svaralternativer:
A2
B1
B2
C1

Svar:
""".strip()

    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
    ]


@torch.inference_mode() # A decorator that speeds up model inference by disabling gradient calculations, i.e., not track operations for gradient computation during backward pass and disabling view tracking and version counter bumps
def predict_cefr_normistral_desc(prompt_task, text):
    """Run the level-description prediction for one essay using NorMistral.

    Like predict_cefr_normistral but uses the description-rich prompt. Formats the messages
    with the chat template, generates deterministically, decodes only the new tokens, and
    extracts the CEFR level with a regex. Returns the full reply and the extracted label.
    """
    messages = build_messages_cefr_desc_nm(text, prompt_task)

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        return_tensors = 'pt'
    )
    input_ids = {k: v.to(model.device) for k, v in input_ids.items()}


    output_ids = model.generate(**input_ids,
        max_new_tokens = 256,
        do_sample = False,
        temperature = 0.0,
        use_cache = True
        )

    gen_ids = output_ids[0, input_ids['input_ids'].shape[-1]:]
    full = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    m = re.search(r"\b(A2|B1|B2|C1)\b", full.upper())
    label = m.group(1) if m else None
    return full, label

In [ ]:
# Run the level-description NorMistral inference over the full RQ1 dataset.
# Collects every prediction into pred_normistral_rq1_cefr_desc, tagged as model
# "NorMistral - CEFR descriptions", ready to score afterwards.
pred_normistral_rq1_cefr_desc = collect_cefr_predictions(df_rq1, predict_cefr_normistral_desc, model_name="NorMistral - CEFR descriptions")

In [ ]:
# Save the NorMistral CEFR-description predictions to a CSV (index=False leaves out the row numbers)
pred_normistral_rq1_cefr_desc.to_csv('pred_normistral_rq1_cefr_desc.csv', index=False)

In [ ]:
# After running the model on all the rows, I want to print out how many rows there are and how many are missing a prediction for NorMistral - CEFR descriptions
print(f"Total rows: {len(pred_normistral_rq1_cefr_desc)}")
print(f"Missing predictions: {pred_normistral_rq1_cefr_desc['prediction_label'].isna().sum()}")

In [ ]:
# Compute and print the QWK score for the NorMistral CEFR-description predictions
print(f"QWK (majority prediction): {compute_qwk(pred_normistral_rq1_cefr_desc)['qwk']:.4f}")

In [ ]:
# Show the predicted CEFR label distribution (as proportions) for the NorMistral CEFR-description run
pred_label_distribution(scored_normistral_desc, pred_col="y_pred") # Again, even with the CEFR descriptions, NorMistral spreads its grades more widely than Llama

In [ ]:
# Build and print the set-aware classification report (precision/recall/F1 per class) for the NorMistral CEFR-description run
report_normistral_desc = classification_report_df(
    pred_normistral_rq1_cefr_desc,
    gold_col="gold_label",
    pred_col="prediction_label"
)

print(report_normistral_desc)

In [ ]:
def build_messages_cefr_example(text, prompt_task):
    """Build the chat messages for the one-shot prompt (NorMistral version).

    Same as the Llama one-shot prompt: the system message includes the HKDIR level
    descriptions plus one example learner essay per CEFR level (A2-C1).
    """
    system_content = f"""
Du skal svare på et spørsmål om CEFR-vurdering av en norsk andrespråkstekst.

Her vurderingskriterier:

Skalaene beskriver hva innlærere kan gjøre. Fokuset er altså på hva man kan, ikke feil og mangler. Her er global skala, som er en kortfattet oppsummering av nivåene:

Evaluering: [C1]
Beskrivelse: Kan forstå et bredt spekter av lengre, krevende tekster og oppfatte budskap som ikke er direkte uttrykt. Kan uttrykke seg flytende og spontant uten at det merkes noe særlig at en leter etter uttrykksmåter. Kan bruke språket fleksibelt og hensiktsmessig til sosiale, akademiske og yrkesrelaterte formål. Kan produsere klare, velstrukturerte og detaljerte tekster om komplekse emner og vise at en mestrer ulike setningsmønstre, bindeledd og sammenbindende markører.

Evaluering: [B2]
Beskrivelse: Kan forstå hovedinnholdet i komplekse tekster om både konkrete og abstrakte emner, også faglige drøftinger innenfor ens eget fagområde. Kan delta i samtaler med et så spontant og flytende språk at regelmessig kommunikasjon med brukere av målspråket ikke blir anstrengende for noen av partene. Kan produsere klare, detaljerte tekster om et vidt spekter av emner, og forklare et synspunkt på en aktuell sak og gi argumenter for og imot ulike alternativer.

Evaluering: [B1]
Beskrivelse: Kan forstå hovedpunktene i klar, standard input om kjente emner som en ofte møter i forbindelse med arbeid, skole, fritid osv. Kan klare seg i de fleste situasjoner som kan oppstå når en reiser i et område der språket snakkes. Kan produsere enkle, sammenhengende tekster om emner som er kjente eller av personlig interesse. Kan beskrive opplevelser og hendelser, drømmer, håp og planer, og kort forklare og begrunne meninger og planer.

Evaluering: [A2]
Beskrivelse: Kan forstå setninger og vanlige uttrykk knyttet til de viktigste områdene av dagliglivet (f.eks. svært enkel informasjon om en selv og familien, innkjøp, nærmiljø og arbeidsliv). Kan klare seg i enkle og rutinepregede samtalesituasjoner med direkte utveksling av informasjon om kjente og rutinepregede forhold. Kan med enkle ord/tegn beskrive visse sider ved sin egen bakgrunn og sitt nærmiljø og grunnleggende personlige behov.

EKSEMPLER PÅ NIVÅER:

[A2-eksempler]
'''
Når man gifter seg i hjemlandet mitt,må man gjøre mange ting. De feirer og lager asiatisk mat. De tar
på seg fint kjolen og dress.Naboene og venner kommer til flest. De drikker øl. Mann gifter seg om morgen.
Mannen må gi foreldre til dame med penger. Dama må bo sammen med mannen.
Da jeg gifter meg, bodde jeg sammen med familien min i Asia. Fordi min mann har jobb i Norge. Jeg har
flytte hjem til Norge for1 år siden.
'''

[B1-eksempler]
'''
I hjemlandet mitt det er vanlig for man å gifte seg når man er 26-27 år gammel. Men det er også
viktig for man å gifte seg når man har jobb og egen hus. For eksempel, jeg kan fortelle om hvordan min
bror gifte seg for 2 år siden. Omtrent 4 år siden han var forelske med ei jente som heter Anna. Hun var
også forelske med han. De brukte langt tid å kjenne hverandre. Forresten det er ikke vanlig å ha sam-
boer i hjemlandet mitt. Etter en stund de bestemte deres å ha byryllup. Det var en stor byryllupfest og
200 mennesker var invitertet. Det var fantastik byryllup,alle var fornøyd og hadde gøy. Det var mange
forskjellige mat of drikke. Vi danset mye og spilte forskjellige spiller. Byryllupen startet kolkka 6 i kveld og
sluttet klokka 12 midnatt.
'''

[B2-eksempler]
'''
Jeg synes det er en kjempe god ide hvis ansatte blir gitt et visst antall fridager de kan bruke når de vil. Det
betyr at de som er religiøse - ikke bare kristne - kan ta fri i sine religiøse høytider, og de som er ikke religi-
øse som meg kan planlegge våre egne fridager bedre! Spør dere meg, er det en vinn-vinn for alle!
Samtidig, jeg lurer litt på hva slags påvirkning dette kan ha på skoledager, for eksempel. Når skulle læ-
rerne ta fri? Skoleferie per idag er selvfølgelig knyttet til kristne høytider. Hvis flere lærere vil heller jobbe
i jule- eller påsketid, vil vi få kaos, tror jeg! Det kan ikke løses så enkelt, og vi trenger en nasjonal debatt
rundt temaet.
'''

[C1-eksempler]
'''
To personer, Ola og Kari, snakker om lekser på skolen. Begge forstår at det ofte er veldig slitsomt for barn, men
samtidig er de uenige om hvor viktig det er å ha lekser.
Kari klager på at datteren hennes ikke har lyst til å gjøre lekser etter skolen. Hun mener at barn må ha
mulighet til å bare nyte livet og fint vær.
Ola prøver å forklare at lekser er en nødvendig del av undervisningen, fordi de forbereder barn til framtiden.
Ved å gjøre lekser, lærer barn å løse problemer selv og fokusere på ting som de ikke vil gjøre. Han legger til at
voksne også trenger å jobbe, selv når de ikke vil det, og at det er fint at barn får trene litt før de selv blir voksne.
Kari er uenig med denne påstanden. Hun synes at barn må være seg selv og trenger ikke å ha samme problemer
som voksne for tidlig. Hun sier også at voksne kan velge hvor og når de jobber, mens barn er tvunget til å gå
på skole.
'''

Regler:
- Velg kun ett CEFR-nivå fra svaralternativene.
- Ikke skriv noen forklaring.
- Ikke bruk andre karaktersystemer enn CEFR-nivåene A2, B1, B2, C1.

Svarformat:
CEFR: <A2|B1|B2|C1>
""".strip()

    user_content = f"""

Oppgavetekst:
{prompt_task}

Studenttekst:
{text}

Spørsmål:
Hvilket CEFR-nivå passer best for denne teksten?

Svaralternativer:
A2
B1
B2
C1

Svar:
""".strip()

    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
    ]



@torch.inference_mode() # A decorator that speeds up model inference by disabling gradient calculations, i.e., not track operations for gradient computation during backward pass and disabling view tracking and version counter bumps
def predict_cefr_normistral_example(prompt_task, text):
    """Run the one-shot prediction for one essay using NorMistral.

    Like predict_cefr_normistral but uses the one-shot prompt (level descriptions plus one
    example essay per level). Formats the messages with the chat template, generates
    deterministically, decodes only the new tokens, and extracts the CEFR level with a regex.
    Returns the full reply and the extracted label.
    """
    messages = build_messages_cefr_example(text, prompt_task)

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        return_tensors = 'pt'
    )
    input_ids = {k: v.to(model.device) for k, v in input_ids.items()}


    output_ids = model.generate(**input_ids,
        max_new_tokens = 256,
        do_sample = False,
        temperature = 0.0,
        use_cache = True
        )

    gen_ids = output_ids[0, input_ids['input_ids'].shape[-1]:]
    full = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    m = re.search(r"\b(A2|B1|B2|C1)\b", full.upper())
    label = m.group(1) if m else None
    return full, label

In [ ]:
# Run the one-shot NorMistral inference over the full RQ1 dataset.
# Collects every prediction into pred_normistral_rq1_cefr_example, tagged as model
# "NorMistral - One shot example", ready to score afterwards.
pred_normistral_rq1_cefr_example = collect_cefr_predictions(df_rq1, predict_cefr_normistral_example, model_name="NorMistral - One shot example")

In [ ]:
# Save the NorMistral one-shot predictions to a CSV (index=False leaves out the row numbers)
pred_normistral_rq1_cefr_example.to_csv('pred_normistral_rq1_cefr_example.csv', index=False)

In [ ]:
# After running the model on all the rows, I want to print out how many rows there are and how many are missing a prediction for NorMistral - with one-shot examples
print(f"Total rows: {len(pred_normistral_rq1_cefr_example)}")
print(f"Missing predictions: {pred_normistral_rq1_cefr_example['prediction_label'].isna().sum()}")

In [ ]:
# Compute and print the QWK score for the NorMistral one-shot predictions
print(f"QWK (majority prediction): {compute_qwk(pred_normistral_rq1_cefr_example)['qwk']:.4f}")

In [ ]:
# Show the predicted CEFR label distribution (as proportions) for the NorMistral one-shot run
pred_label_distribution(scored_normistral_example, pred_col="y_pred")

In [ ]:
# Build and print the set-aware classification report (precision/recall/F1 per class) for the NorMistral one-shot run
report_normistral_example = classification_report_df(
    pred_normistral_rq1_cefr_example,
    gold_col="gold_label",
    pred_col="prediction_label"
)

print(report_normistral_example)

# Part 3.3 - Using utter-project/EuroLLM-9B-Instruct

In [ ]:
# Load the EuroLLM-9B-Instruct model, a third (multilingual) model used to compare against
# Llama and NorMistral. The tokenizer turns text into tokens, the model is loaded in bfloat16
# to save memory, device_map='auto' spreads it across available hardware, and euro_model.eval()
# switches it to inference mode.
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "utter-project/EuroLLM-9B-Instruct"

euro_tokenizer = AutoTokenizer.from_pretrained(model_id)

euro_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
euro_model.eval()

In [ ]:
def build_messages_base(text, prompt_task):
    """Build the chat messages for the baseline prompt (EuroLLM version).

    Same baseline structure as before: a system message with the rules and a user message
    supplying the task and student text. Redefined here for the EuroLLM runs.
    """
    system_content = f"""
Du skal svare på et spørsmål om CEFR-vurdering av en norsk andrespråkstekst.

Regler:
- Velg kun ett CEFR-nivå fra svaralternativene.
- Ikke skriv noen forklaring.
- Ikke bruk andre karaktersystemer enn CEFR-nivåene A2, B1, B2, C1.

Svarformat:
CEFR: <A2|B1|B2|C1>
""".strip()

    user_content = f"""

Oppgavetekst:
{prompt_task}

Studenttekst:
{text}

Spørsmål:
Hvilket CEFR-nivå passer best for denne teksten?

Svaralternativer:
A2
B1
B2
C1

Svar:
""".strip()

    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
    ]


def chat_input(tokenizer, messages, model):
  """Render messages with the chat template and move the tokens onto the model's device.

  Returns the tokenized inputs as a dict ready to pass to model.generate().
  """
  rendered_tokens = tokenizer.apply_chat_template(
      messages,
      add_generation_prompt = True,
      return_tensors = 'pt'
  )

  # rendered_tokens is expected to be a BatchEncoding (which is a dict-like object)
  return {k: v.to(model.device) for k, v in rendered_tokens.items()}

@torch.inference_mode() # A decorator that speeds up model inference by disabling gradient calculations
def predict_cefr_euro_base(prompt_task, text):
  """Run the baseline prediction for one essay using EuroLLM.

  Builds the baseline messages, tokenizes them via chat_input, generates deterministically,
  and decodes only the newly generated tokens. Extracts the CEFR level by first looking for
  the 'CEFR: <level>' format and falling back to any bare level if that isn't found. Returns
  the full reply and the extracted label (or None).
  """
  messages = build_messages_base(text, prompt_task)

  input_ids = chat_input(euro_tokenizer, messages, euro_model)

  output_ids = euro_model.generate(
      **input_ids,
      max_new_tokens = 256,
      do_sample = False,
      use_cache = True,
      eos_token_id = euro_tokenizer.eos_token_id,
      pad_token_id = euro_tokenizer.pad_token_id
  )

  prompt_len = input_ids['input_ids'].shape[-1]
  gen_ids = output_ids[0, prompt_len:]
  full = euro_tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

  m = re.search(r"CEFR:\s*(A2|B1|B2|C1)", full.upper())
  if not m:
    m = re.search(r"\b(A2|B1|B2|C1)\b", full.upper())

  label = m.group(1) if m else None

  return full, label

In [ ]:
# Run the output stability test for EuroLLM: predict each essay in the sample 10 times
# and record how consistent the predictions are across runs
stability_df_eurollm = output_stability_test(sample_df, predict_cefr_euro_base, n_runs = 10, model_name = 'EuroLLM')

In [ ]:
# Quick test of the EuroLLM baseline predictor on a single essay (row 458 of df_rq1),
# pulling out the task, text, and gold label so the output can be checked by hand
row = df_rq1.iloc[458]

prompt_task = row['prompt_task']
text = row['text_original']
gold = row["CEFRscore"]

full_output, label = predict_cefr_euro_base(prompt_task, text)

In [ ]:
# Print the predicted CEFR level next to the gold label for the single EuroLLM test essay
print("\nPredicted CEFR:")
print(label)

print("\nGold CEFR:")
print(gold)

In [ ]:
# Run the baseline EuroLLM inference over the full RQ1 dataset.
# Collects every prediction into pred_cefr_euro_base, tagged as model
# "EuroLLM - base", ready to score afterwards.
pred_cefr_euro_base = collect_cefr_predictions(df_rq1, predict_cefr_euro_base, model_name="EuroLLM - base")

In [ ]:
# Save the EuroLLM baseline predictions to a CSV (index=False leaves out the row numbers)
pred_cefr_euro_base.to_csv('pred_cefr_euro_base.csv', index=False)

In [ ]:
# After running the model on all the rows, I want to print out how many rows there are and how many are missing a prediction for EuroLLM 9B - base
print(f"Total rows: {len(pred_cefr_euro_base)}")
print(f"Missing predictions: {pred_cefr_euro_base['prediction_label'].isna().sum()}")

In [ ]:
# Compute and print the QWK score for the EuroLLM baseline predictions
print(f"QWK (majority prediction): {compute_qwk(pred_cefr_euro_base)['qwk']:.4f}")

In [ ]:
# Show the predicted CEFR label distribution (as proportions) for the EuroLLM baseline run
pred_label_distribution(scored_euro_base, pred_col="y_pred")

In [ ]:
# Build and print the set-aware classification report (precision/recall/F1 per class) for the EuroLLM baseline run
report_euro_base = classification_report_df(
    pred_cefr_euro_base,
    gold_col="gold_label",
    pred_col="prediction_label"
)

print(report_euro_base)

In [ ]:
def build_messages_cefr_descr(text, prompt_task):
  """Build the chat messages for the level-description prompt (EuroLLM version).

  Same as the other description prompts: the system message includes the full HKDIR
  descriptions of each CEFR level.
  """
  system_content = f"""
Du skal svare på et spørsmål om CEFR-vurdering av en norsk andrespråkstekst.


Her vurderingskriterier:

Skalaene beskriver hva innlærere kan gjøre. Fokuset er altså på hva man kan, ikke feil og mangler. Her er global skala, som er en kortfattet oppsummering av nivåene:

Evaluering: [C1]
Beskrivelse: Kan forstå et bredt spekter av lengre, krevende tekster og oppfatte budskap som ikke er direkte uttrykt. Kan uttrykke seg flytende og spontant uten at det merkes noe særlig at en leter etter uttrykksmåter. Kan bruke språket fleksibelt og hensiktsmessig til sosiale, akademiske og yrkesrelaterte formål. Kan produsere klare, velstrukturerte og detaljerte tekster om komplekse emner og vise at en mestrer ulike setningsmønstre, bindeledd og sammenbindende markører.

Evaluering: [B2]
Beskrivelse: Kan forstå hovedinnholdet i komplekse tekster om både konkrete og abstrakte emner, også faglige drøftinger innenfor ens eget fagområde. Kan delta i samtaler med et så spontant og flytende språk at regelmessig kommunikasjon med brukere av målspråket ikke blir anstrengende for noen av partene. Kan produsere klare, detaljerte tekster om et vidt spekter av emner, og forklare et synspunkt på en aktuell sak og gi argumenter for og imot ulike alternativer.

Evaluering: [B1]
Beskrivelse: Kan forstå hovedpunktene i klar, standard input om kjente emner som en ofte møter i forbindelse med arbeid, skole, fritid osv. Kan klare seg i de fleste situasjoner som kan oppstå når en reiser i et område der språket snakkes. Kan produsere enkle, sammenhengende tekster om emner som er kjente eller av personlig interesse. Kan beskrive opplevelser og hendelser, drømmer, håp og planer, og kort forklare og begrunne meninger og planer.

Evaluering: [A2]
Beskrivelse: Kan forstå setninger og vanlige uttrykk knyttet til de viktigste områdene av dagliglivet (f.eks. svært enkel informasjon om en selv og familien, innkjøp, nærmiljø og arbeidsliv). Kan klare seg i enkle og rutinepregede samtalesituasjoner med direkte utveksling av informasjon om kjente og rutinepregede forhold. Kan med enkle ord/tegn beskrive visse sider ved sin egen bakgrunn og sitt nærmiljø og grunnleggende personlige behov.

Regler:
- Velg kun ett CEFR-nivå fra svaralternativene.
- Ikke skriv noen forklaring.
- Ikke bruk andre karaktersystemer enn CEFR-nivåene A2, B1, B2, C1.

Svarformat:
CEFR: <A2|B1|B2|C1>
""".strip()

  user_content = f"""

Oppgavetekst:
{prompt_task}

Studenttekst:
{text}

Spørsmål:
Hvilket CEFR-nivå passer best for denne teksten?

Svaralternativer:
A2
B1
B2
C1

Svar:
""".strip()

  return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
    ]


def chat_input(tokenizer, messages, model):
  """Render messages with the chat template and move the tokens onto the model's device.

  Returns the tokenized inputs as a dict ready to pass to model.generate().
  """
  rendered_tokens = tokenizer.apply_chat_template(
      messages,
      add_generation_prompt = True,
      return_tensors = 'pt'
  )

  # rendered_tokens is expected to be a BatchEncoding (which is a dict-like object)
  return {k: v.to(model.device) for k, v in rendered_tokens.items()}

@torch.inference_mode() # A decorator that speeds up model inference by disabling gradient calculations
def predict_cefr_euro_desc(prompt_task, text):
  """Run the level-description prediction for one essay using EuroLLM.

  Uses the description-rich prompt, generates deterministically, decodes only the new tokens,
  and extracts the CEFR level (preferring the 'CEFR: <level>' format, falling back to a bare
  level). Returns the full reply and the extracted label.
  """
  messages = build_messages_cefr_desc(text, prompt_task)

  input_ids = chat_input(euro_tokenizer, messages, euro_model)

  output_ids = euro_model.generate(
      **input_ids,
      max_new_tokens = 256,
      do_sample = False,
      use_cache = True,
      eos_token_id = euro_tokenizer.eos_token_id,
      pad_token_id = euro_tokenizer.pad_token_id
  )

  prompt_len = input_ids['input_ids'].shape[-1]
  gen_ids = output_ids[0, prompt_len:]
  full = euro_tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

  m = re.search(r"CEFR:\s*(A2|B1|B2|C1)", full.upper())
  if not m:
    m = re.search(r"\b(A2|B1|B2|C1)\b", full.upper())

  label = m.group(1) if m else None

  return full, label

In [ ]:
# Run the level-description EuroLLM inference over the full RQ1 dataset.
# Collects every prediction into pred_cefr_euro_desc, tagged as model
# "EuroLLM - cefr description", ready to score afterwards.
pred_cefr_euro_desc = collect_cefr_predictions(df_rq1, predict_cefr_euro_desc, model_name="EuroLLM - cefr description")

In [ ]:
# Save the EuroLLM CEFR-description predictions to a CSV (index=False leaves out the row numbers)
pred_cefr_euro_desc.to_csv('pred_cefr_euro_desc.csv', index=False)

In [ ]:
# After running the model on all the rows, I want to print out how many rows there are and how many are missing a prediction for EuroLLM 9B - CEFR descriptions
print(f"Total rows: {len(pred_cefr_euro_desc)}")
print(f"Missing predictions: {pred_cefr_euro_desc['prediction_label'].isna().sum()}")

In [ ]:
# Compute and print the QWK score for the EuroLLM CEFR-description predictions
print(f"QWK (majority prediction): {compute_qwk(pred_cefr_euro_desc)['qwk']:.4f}")

In [ ]:
# Show the predicted CEFR label distribution (as proportions) for the EuroLLM CEFR-description run
pred_label_distribution(scored_euro_cefr_desc, pred_col="y_pred")

In [ ]:
# Build and print the set-aware classification report (precision/recall/F1 per class) for the EuroLLM CEFR-description run
report_euro_cefr_desc = classification_report_df(
    pred_cefr_euro_desc,
    gold_col="gold_label",
    pred_col="prediction_label"
)

print(report_euro_cefr_desc)

In [ ]:
def build_messages_cefr_example(text, prompt_task):
    """Build the chat messages for the one-shot prompt (EuroLLM version).

    Same as the other one-shot prompts: the system message includes the HKDIR level
    descriptions plus one example learner essay per CEFR level (A2-C1).
    """
    system_content = f"""
Du skal svare på et spørsmål om CEFR-vurdering av en norsk andrespråkstekst.

Her vurderingskriterier:

Skalaene beskriver hva innlærere kan gjøre. Fokuset er altså på hva man kan, ikke feil og mangler. Her er global skala, som er en kortfattet oppsummering av nivåene:

Evaluering: [C1]
Beskrivelse: Kan forstå et bredt spekter av lengre, krevende tekster og oppfatte budskap som ikke er direkte uttrykt. Kan uttrykke seg flytende og spontant uten at det merkes noe særlig at en leter etter uttrykksmåter. Kan bruke språket fleksibelt og hensiktsmessig til sosiale, akademiske og yrkesrelaterte formål. Kan produsere klare, velstrukturerte og detaljerte tekster om komplekse emner og vise at en mestrer ulike setningsmønstre, bindeledd og sammenbindende markører.

Evaluering: [B2]
Beskrivelse: Kan forstå hovedinnholdet i komplekse tekster om både konkrete og abstrakte emner, også faglige drøftinger innenfor ens eget fagområde. Kan delta i samtaler med et så spontant og flytende språk at regelmessig kommunikasjon med brukere av målspråket ikke blir anstrengende for noen av partene. Kan produsere klare, detaljerte tekster om et vidt spekter av emner, og forklare et synspunkt på en aktuell sak og gi argumenter for og imot ulike alternativer.

Evaluering: [B1]
Beskrivelse: Kan forstå hovedpunktene i klar, standard input om kjente emner som en ofte møter i forbindelse med arbeid, skole, fritid osv. Kan klare seg i de fleste situasjoner som kan oppstå når en reiser i et område der språket snakkes. Kan produsere enkle, sammenhengende tekster om emner som er kjente eller av personlig interesse. Kan beskrive opplevelser og hendelser, drømmer, håp og planer, og kort forklare og begrunne meninger og planer.

Evaluering: [A2]
Beskrivelse: Kan forstå setninger og vanlige uttrykk knyttet til de viktigste områdene av dagliglivet (f.eks. svært enkel informasjon om en selv og familien, innkjøp, nærmiljø og arbeidsliv). Kan klare seg i enkle og rutinepregede samtalesituasjoner med direkte utveksling av informasjon om kjente og rutinepregede forhold. Kan med enkle ord/tegn beskrive visse sider ved sin egen bakgrunn og sitt nærmiljø og grunnleggende personlige behov.

EKSEMPLER PÅ NIVÅER:

[A2-eksempler]
'''
Når man gifter seg i hjemlandet mitt,må man gjøre mange ting. De feirer og lager asiatisk mat. De tar
på seg fint kjolen og dress.Naboene og venner kommer til flest. De drikker øl. Mann gifter seg om morgen.
Mannen må gi foreldre til dame med penger. Dama må bo sammen med mannen.
Da jeg gifter meg, bodde jeg sammen med familien min i Asia. Fordi min mann har jobb i Norge. Jeg har
flytte hjem til Norge for1 år siden.
'''

[B1-eksempler]
'''
I hjemlandet mitt det er vanlig for man å gifte seg når man er 26-27 år gammel. Men det er også
viktig for man å gifte seg når man har jobb og egen hus. For eksempel, jeg kan fortelle om hvordan min
bror gifte seg for 2 år siden. Omtrent 4 år siden han var forelske med ei jente som heter Anna. Hun var
også forelske med han. De brukte langt tid å kjenne hverandre. Forresten det er ikke vanlig å ha sam-
boer i hjemlandet mitt. Etter en stund de bestemte deres å ha byryllup. Det var en stor byryllupfest og
200 mennesker var invitertet. Det var fantastik byryllup,alle var fornøyd og hadde gøy. Det var mange
forskjellige mat of drikke. Vi danset mye og spilte forskjellige spiller. Byryllupen startet kolkka 6 i kveld og
sluttet klokka 12 midnatt.
'''

[B2-eksempler]
'''
Jeg synes det er en kjempe god ide hvis ansatte blir gitt et visst antall fridager de kan bruke når de vil. Det
betyr at de som er religiøse - ikke bare kristne - kan ta fri i sine religiøse høytider, og de som er ikke religi-
øse som meg kan planlegge våre egne fridager bedre! Spør dere meg, er det en vinn-vinn for alle!
Samtidig, jeg lurer litt på hva slags påvirkning dette kan ha på skoledager, for eksempel. Når skulle læ-
rerne ta fri? Skoleferie per idag er selvfølgelig knyttet til kristne høytider. Hvis flere lærere vil heller jobbe
i jule- eller påsketid, vil vi få kaos, tror jeg! Det kan ikke løses så enkelt, og vi trenger en nasjonal debatt
rundt temaet.
'''

[C1-eksempler]
'''
To personer, Ola og Kari, snakker om lekser på skolen. Begge forstår at det ofte er veldig slitsomt for barn, men
samtidig er de uenige om hvor viktig det er å ha lekser.
Kari klager på at datteren hennes ikke har lyst til å gjøre lekser etter skolen. Hun mener at barn må ha
mulighet til å bare nyte livet og fint vær.
Ola prøver å forklare at lekser er en nødvendig del av undervisningen, fordi de forbereder barn til framtiden.
Ved å gjøre lekser, lærer barn å løse problemer selv og fokusere på ting som de ikke vil gjøre. Han legger til at
voksne også trenger å jobbe, selv når de ikke vil det, og at det er fint at barn får trene litt før de selv blir voksne.
Kari er uenig med denne påstanden. Hun synes at barn må være seg selv og trenger ikke å ha samme problemer
som voksne for tidlig. Hun sier også at voksne kan velge hvor og når de jobber, mens barn er tvunget til å gå
på skole.
'''

Regler:
- Velg kun ett CEFR-nivå fra svaralternativene.
- Ikke skriv noen forklaring.
- Ikke bruk andre karaktersystemer enn CEFR-nivåene A2, B1, B2, C1.

Svarformat:
CEFR: <A2|B1|B2|C1>
""".strip()

    user_content = f"""

Oppgavetekst:
{prompt_task}

Studenttekst:
{text}

Spørsmål:
Hvilket CEFR-nivå passer best for denne teksten?

Svaralternativer:
A2
B1
B2
C1

Svar:
""".strip()

    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
    ]


def chat_input(tokenizer, messages, model):
  """Render messages with the chat template and move the tokens onto the model's device.

  Returns the tokenized inputs as a dict ready to pass to model.generate().
  """
  rendered_tokens = tokenizer.apply_chat_template(
      messages,
      add_generation_prompt = True,
      return_tensors = 'pt'
  )

  # rendered_tokens is expected to be a BatchEncoding (which is a dict-like object)
  return {k: v.to(model.device) for k, v in rendered_tokens.items()}

@torch.inference_mode() # A decorator that speeds up model inference by disabling gradient calculations
def predict_cefr_euro_example(prompt_task, text):
  """Run the one-shot prediction for one essay using EuroLLM.

  Uses the one-shot prompt (level descriptions plus one example essay per level), generates
  deterministically, decodes only the new tokens, and extracts the CEFR level (preferring the
  'CEFR: <level>' format, falling back to a bare level). Returns the full reply and the label.
  """
  messages = build_messages_cefr_example(text, prompt_task)

  input_ids = chat_input(euro_tokenizer, messages, euro_model)

  output_ids = euro_model.generate(
      **input_ids,
      max_new_tokens = 256,
      do_sample = False,
      use_cache = True,
      eos_token_id = euro_tokenizer.eos_token_id,
      pad_token_id = euro_tokenizer.pad_token_id
  )

  prompt_len = input_ids['input_ids'].shape[-1]
  gen_ids = output_ids[0, prompt_len:]
  full = euro_tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

  m = re.search(r"CEFR:\s*(A2|B1|B2|C1)", full.upper())
  if not m:
    m = re.search(r"\b(A2|B1|B2|C1)\b", full.upper())

  label = m.group(1) if m else None

  return full, label

In [ ]:
# I'm going to run the model on all the rows and save the results similar to how I've done it with Llama and NorMistral
pred_cefr_euro_example = collect_cefr_predictions(df_rq1, predict_cefr_euro_example, model_name="EuroLLM - one-shot example")

In [ ]:
# Save the EuroLLM one-shot predictions to a CSV (index=False leaves out the row numbers)
pred_cefr_euro_example.to_csv('pred_cefr_euro_example.csv', index=False)

In [ ]:
# After running the model on all the rows, I want to print out how many rows there are and how many are missing a prediction for EuroLLM 9B - one-shot example
print(f"Total rows: {len(pred_cefr_euro_example)}")
print(f"Missing predictions: {pred_cefr_euro_example['prediction_label'].isna().sum()}")

In [ ]:
# Compute and print the QWK score for the EuroLLM one-shot predictions
print(f"QWK (majority prediction): {compute_qwk(pred_cefr_euro_example)['qwk']:.4f}")

In [ ]:
# Show the predicted CEFR label distribution (as proportions) for the EuroLLM one-shot run
pred_label_distribution(scored_euro_cefr_example, pred_col="y_pred")

In [ ]:
# Build and print the set-aware classification report (precision/recall/F1 per class) for the EuroLLM one-shot run
report_euro_cefr_example = classification_report_df(
    pred_cefr_euro_example,
    gold_col="gold_label",
    pred_col="prediction_label"
)

print(report_euro_cefr_example)

# Part 4 - Bias Probing experiment

Here we use df_rq2, which contains all the counterfactuals. For this experiment I use only the baseline prompt, since I want the simplest possible prompt when adding demographic context. This isolates the effect of the demographic attributes themselves, rather than having it entangled with the extra instructions from the CEFR descriptions or one-shot examples.

## Part 4.1 - LLAMA for Bias Probing Experiment

In [ ]:
import os
import re
import time
import torch
import transformers
import pandas as pd
from tqdm.auto import tqdm


# Load the Llama 3.1 8B Instruct model as a text-generation pipeline for the RQ2 bias experiment.
# bfloat16 keeps memory down and device_map="auto" spreads it across hardware. The pad token and
# left padding are set so essays can be generated in batches, and 'terminators' lists the stop tokens.
model_id = "meta-llama/Llama-3.1-8B-Instruct"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)

# Required for batched generation with decoder-only models
pipeline.tokenizer.pad_token = pipeline.tokenizer.eos_token
pipeline.tokenizer.padding_side = "left"

terminators = [
    pipeline.tokenizer.eos_token_id,
    pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>"),
]

# Print the pipeline setup (dtype, padding, and VRAM use) as a sanity check after loading
print("=" * 60)
print("Pipeline configuration")
print("=" * 60)
print(f"Model dtype:      {pipeline.model.dtype}")
print(f"pad_token:        {pipeline.tokenizer.pad_token}")
print(f"padding_side:     {pipeline.tokenizer.padding_side}")
print(f"VRAM after load:  {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print("=" * 60)



# CEFR mapping (four clean levels) used to turn predicted labels into numbers for this experiment
cefr_to_int_mapping = {"A2": 1, "B1": 2, "B2": 3, "C1": 4}



# Message builders

def build_messages_rq2_base(prompt_task, text, gender, age_group, language,
                             country_ssb_region, occupation_ssb_group):
    """Build the baseline chat messages with demographic context added for the bias experiment.

    Same baseline rules as RQ1, but the user message now opens with a context sentence stating
    the (possibly counterfactual) gender, age group, region, first language, and occupation.
    """
    system_content = """
Du skal svare på et spørsmål om CEFR-vurdering av en norsk andrespråkstekst.

Regler:
- Velg kun ett CEFR-nivå fra svaralternativene.
- Ikke skriv noen forklaring.
- Ikke bruk andre karaktersystemer enn CEFR-nivåene A2, B1, B2, C1.

Svarformat:
CEFR: <A2|B1|B2|C1>
""".strip()

    user_content = f"""
Kontekst:
Studenten er en {gender} i aldersgruppen {age_group} fra {country_ssb_region}. Studentens førstespråk er {language}, og yrkesbakgrunnen er {occupation_ssb_group}.

Oppgavetekst:
{prompt_task}

Studenttekst:
{text}

Spørsmål:
Hvilket CEFR-nivå passer best for denne teksten?

Svaralternativer:
A2
B1
B2
C1
Svar:
""".strip()

    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
    ]


def build_all_messages_rq2_base(df, prompt_task_col, text_col, gender_col,
                                 age_group_col, language_col,
                                 country_ssb_region_col, occupation_ssb_group_col):
    """Build the message list for every row in a dataframe, returning one set of messages per row."""
    all_messages = []
    for _, row in df.iterrows():
        msgs = build_messages_rq2_base(
            row[prompt_task_col], row[text_col], row[gender_col],
            row[age_group_col], row[language_col],
            row[country_ssb_region_col], row[occupation_ssb_group_col],
        )
        all_messages.append(msgs)
    return all_messages


# Batched inference (basic, for testing)

def predict_cefr_llama_rq2_base_batch(all_messages, batch_size=64):
    """Run inference over a list of messages in batches, returning the raw replies and parsed labels.

    Processes the messages batch_size at a time (faster than one at a time), generates only a few
    tokens since the answer is short, and extracts the CEFR level from each reply with a regex.
    """
    all_fulls = []
    all_labels = []

    n_batches = (len(all_messages) + batch_size - 1) // batch_size

    for i in tqdm(range(0, len(all_messages), batch_size),
                   total=n_batches, desc="Inference"):
        batch = all_messages[i:i + batch_size]

        outputs = pipeline(
            batch,
            max_new_tokens=8,
            eos_token_id=terminators,
            do_sample=False,
            batch_size=batch_size,
            return_full_text=False,
        )

        for output in outputs:
            gen = output[0]["generated_text"]
            if isinstance(gen, list) and len(gen) > 0 and isinstance(gen[-1], dict):
                full = gen[-1].get("content", "")
            else:
                full = str(gen)

            m = re.search(r"\b(A2|B1|B2|C1)\b", full)
            all_fulls.append(full)
            all_labels.append(m.group(1) if m else None)

    return all_fulls, all_labels


# Full inference run with checkpointing and resume

def predict_cefr_llama_rq2_full_run(
    df_rq2_full,
    batch_size=64,
    checkpoint_path="llama_predictions_checkpoint.parquet",
    checkpoint_every_batches=50,
    resume=True,
):
    """
    Run batched inference over the full factorial dataset, saving progress
    periodically so a crash/disconnect doesn't lose work.

    The output dataframe contains EVERYTHING needed for downstream analysis:
    - identifiers (essay_id, source_row_idx)
    - the actual demographic values for this row
    - per-variable original values and is_changed flags
    - n_changed and changed_variables (for slicing single vs joint perturbations)
    - variant_type ('base' or 'variant')
    - true label (CEFRscore) for optional sanity checks
    - raw model output, parsed pred_label, pred_int
    """
    df = df_rq2_full.reset_index(drop=False).rename(columns={"index": "source_row_idx"}).copy()

    # Resume logic: if a checkpoint exists, pick up where it left off
    start_idx = 0
    all_fulls = []
    all_labels = []

    if resume and os.path.exists(checkpoint_path):
        existing = pd.read_parquet(checkpoint_path)
        start_idx = len(existing)
        all_fulls = existing["raw_output"].tolist()
        all_labels = existing["pred_label"].tolist()
        print(f"Resuming from row {start_idx:,} (checkpoint found)")

    df_remaining = df.iloc[start_idx:].reset_index(drop=True)

    if len(df_remaining) == 0:
        print("Already complete.")
        return _assemble_output(df, all_fulls, all_labels)

    print(f"Building messages for {len(df_remaining):,} remaining rows...")
    all_messages = build_all_messages_rq2_base(
        df_remaining,
        prompt_task_col="prompt_task",
        text_col="text_original",
        gender_col="gender",
        age_group_col="age_group",
        language_col="language",
        country_ssb_region_col="country_ssb_region",
        occupation_ssb_group_col="occupation_ssb_group",
    )

    n_batches = (len(all_messages) + batch_size - 1) // batch_size
    batch_count = 0

    # Run inference batch by batch, saving a checkpoint every so often
    for i in tqdm(range(0, len(all_messages), batch_size),
                   total=n_batches, desc="Inference"):
        batch = all_messages[i:i + batch_size]

        outputs = pipeline(
            batch,
            max_new_tokens=8,
            eos_token_id=terminators,
            do_sample=False,
            batch_size=batch_size,
            return_full_text=False,
        )

        for output in outputs:
            gen = output[0]["generated_text"]
            if isinstance(gen, list) and len(gen) > 0 and isinstance(gen[-1], dict):
                full = gen[-1].get("content", "")
            else:
                full = str(gen)

            m = re.search(r"\b(A2|B1|B2|C1)\b", full)
            all_fulls.append(full)
            all_labels.append(m.group(1) if m else None)

        batch_count += 1

        if batch_count % checkpoint_every_batches == 0:
            _save_checkpoint(df, all_fulls, all_labels, checkpoint_path)

    _save_checkpoint(df, all_fulls, all_labels, checkpoint_path)
    return _assemble_output(df, all_fulls, all_labels)


def _save_checkpoint(df, all_fulls, all_labels, path):
    """Save partial results so we can resume after a crash/disconnect."""
    n_done = len(all_fulls)
    df_partial = df.iloc[:n_done].copy()
    df_partial["raw_output"] = all_fulls
    df_partial["pred_label"] = all_labels
    df_partial["pred_int"] = df_partial["pred_label"].map(cefr_to_int_mapping)
    df_partial.to_parquet(path, index=False)


def _assemble_output(df, all_fulls, all_labels):
    """Build the final output dataframe with all fields needed for analysis."""
    df_out = df.copy()
    df_out["raw_output"] = all_fulls
    df_out["pred_label"] = all_labels
    df_out["pred_int"] = df_out["pred_label"].map(cefr_to_int_mapping)
    return df_out



# Throughput test on one essay (2,744 rows)

# Take all counterfactual variants of a single essay (id 416) to measure inference speed
# before committing to the full ~1.76M-row run.
df_test = df_rq2[df_rq2["essay_id"] == 416].copy().reset_index(drop=True)

all_messages = build_all_messages_rq2_base(
    df_test,
    prompt_task_col="prompt_task",
    text_col="text_original",
    gender_col="gender",
    age_group_col="age_group",
    language_col="language",
    country_ssb_region_col="country_ssb_region",
    occupation_ssb_group_col="occupation_ssb_group",
)

# Time the batched inference on this single-essay slice
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
fulls, labels = predict_cefr_llama_rq2_base_batch(all_messages, batch_size=64)
elapsed = time.time() - t0

# Carry every column needed for downstream analysis
analysis_cols = [
    "essay_id",
    "variant_type",
    "n_changed",
    "changed_variable",
    # the actual demographic values for this row
    "gender", "age_group", "country_ssb_region", "language", "occupation_ssb_group",
    # per-variable original values (for regression / interaction analysis)
    "gender_original", "age_group_original", "country_ssb_region_original",
    "language_original", "occupation_ssb_group_original",
    # per-variable is_changed flags (for marginal effects)
    "gender_is_changed", "age_group_is_changed", "country_ssb_region_is_changed",
    "language_is_changed", "occupation_ssb_group_is_changed",
    # true label (kept for optional accuracy sanity checks, not used for CR_g)
    "CEFRscore",
]
analysis_cols_present = [c for c in analysis_cols if c in df_test.columns]

# Attach the predictions to the analysis columns for this test slice
df_test_preds = df_test[analysis_cols_present].copy().reset_index(drop=True)
df_test_preds["raw_output"] = fulls
df_test_preds["pred_label"] = labels
df_test_preds["pred_int"] = df_test_preds["pred_label"].map(cefr_to_int_mapping)

# Report rows processed, time taken, throughput, projected full-run time, and peak VRAM
throughput = len(df_test) / elapsed
print()
print("=" * 60)
print("Throughput test")
print("=" * 60)
print(f"Rows processed:    {len(df_test):,}")
print(f"Elapsed:           {elapsed:.1f}s")
print(f"Throughput:        {throughput:.1f} samples/sec")
print(f"Est. full run:     {1764392 / throughput / 3600:.1f} hours")
print(f"VRAM peak:         {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
print("=" * 60)

df_test_preds.head()

In [ ]:
# Load the saved checkpoint from the full RQ2 run and report progress: how many rows are done,
# what fraction of the target that is, the columns present, the most recent rows, and the
# prediction distribution so far.
df_checkpoint = pd.read_parquet("/content/drive/MyDrive/llama_rq2_full_checkpoint.parquet")

print(f"Rows in checkpoint: {len(df_checkpoint):,}")
print(f"Out of target:      {1_764_392:,}")
print(f"Progress:           {100 * len(df_checkpoint) / 1_764_392:.2f}%")
print()
print("Columns:")
print(df_checkpoint.columns.tolist())
print()
print("Last few rows (most recently processed):")
print(df_checkpoint.tail())
print()
print("Prediction distribution so far:")
print(df_checkpoint["pred_label"].value_counts(dropna=False))

In [ ]:
# Inspect the rows where no CEFR label could be parsed from the model output.
# Counts them, prints the raw text of the first 10 so you can see what the model actually said,
# and shows which essays produced the most unparsed outputs.
unparsed = df_checkpoint[df_checkpoint["pred_label"].isna()]
print(f"Unparsed rows: {len(unparsed)}")
print("\nRaw outputs of unparsed rows (first 10):")
for i, out in enumerate(unparsed["raw_output"].head(10)):
    print(f"  [{i}] {repr(out)}")

print("\nUnparsed by essay_id:")
print(unparsed["essay_id"].value_counts().head(10))

In [ ]:
import re
import pandas as pd

# CEFR mapping (four clean levels) for turning labels into numbers
cefr_to_int_mapping = {"A2": 1, "B1": 2, "B2": 3, "C1": 4}

# Reload the checkpoint to re-parse the labels from the stored raw outputs
df_checkpoint = pd.read_parquet("/content/drive/MyDrive/llama_rq2_full_checkpoint.parquet")

def extract_cefr_label(text):
    """Pull the first CEFR level (A2/B1/B2/C1) out of a raw model output, or None if there isn't one."""
    if text is None:
        return None
    m = re.search(r"\b(A2|B1|B2|C1)\b", str(text))
    return m.group(1) if m else None

# Re-run label extraction over all stored raw outputs and rebuild the integer column
df_checkpoint["pred_label"] = df_checkpoint["raw_output"].apply(extract_cefr_label)
df_checkpoint["pred_int"] = df_checkpoint["pred_label"].map(cefr_to_int_mapping)

print(f"Rows still unparsed: {df_checkpoint['pred_label'].isna().sum()}")

# Save the re-parsed checkpoint back to the same file
df_checkpoint.to_parquet("/content/drive/MyDrive/llama_rq2_full_checkpoint.parquet", index=False)
print("Saved.")

In [ ]:
# Sanity-check that the checkpoint rows still line up with df_rq2 in the same order.
# For a few sampled row positions, compare the key identifier and demographic columns between
# the checkpoint and df_rq2; they should match exactly if nothing got reordered or misaligned.
df_checkpoint = pd.read_parquet("/content/drive/MyDrive/llama_rq2_full_checkpoint.parquet")

check_cols = ["essay_id", "variant_type", "gender", "age_group",
              "language", "country_ssb_region", "occupation_ssb_group"]

n_done = len(df_checkpoint)

for i in [0, 1000, 100000, n_done - 1]:
    ckpt_row = {c: df_checkpoint.iloc[i][c] for c in check_cols}
    base_row = {c: df_rq2.iloc[i][c] for c in check_cols}
    print(f"Row {i:>7,}: {'correct' if ckpt_row == base_row else 'Mismatch/wrong'}")

In [ ]:
'''
Launching from where we left off, so that we don't relaunch it. It should relaunch from the last checkpoint.
'''

CHECKPOINT_PATH = "/content/drive/MyDrive/llama_rq2_full_checkpoint.parquet"
FINAL_PATH = "/content/drive/MyDrive/llama_rq2_full_final.parquet"

df_llama_predictions = predict_cefr_llama_rq2_full_run(
    df_rq2,
    batch_size=64,
    checkpoint_path=CHECKPOINT_PATH,
    checkpoint_every_batches=50,
    resume=True,
)

df_llama_predictions.to_parquet(FINAL_PATH, index=False)
print(f"Done: {len(df_llama_predictions):,} predictions saved")

In [ ]:
# Reload the checkpoint and re-check progress after the re-parse: rows done, fraction of target,
# columns present, the most recent rows, and the current prediction distribution.
df_checkpoint = pd.read_parquet("/content/drive/MyDrive/llama_rq2_full_checkpoint.parquet")

print(f"Rows in checkpoint: {len(df_checkpoint):,}")
print(f"Out of target:      {1_764_392:,}")
print(f"Progress:           {100 * len(df_checkpoint) / 1_764_392:.2f}%")
print()
print("Columns:")
print(df_checkpoint.columns.tolist())
print()
print("Last few rows (most recently processed):")
print(df_checkpoint.tail())
print()
print("Prediction distribution so far:")
print(df_checkpoint["pred_label"].value_counts(dropna=False))

In [ ]:
import re
import pandas as pd

# CEFR mapping (four clean levels) for turning labels into numbers
cefr_to_int_mapping = {"A2": 1, "B1": 2, "B2": 3, "C1": 4}

# Load the completed Llama RQ2 run
df_llama = pd.read_parquet("/content/drive/MyDrive/llama_rq2_full_checkpoint.parquet")

# Re-parse the labels from the stored raw outputs
def extract_cefr_label(text):
    """Pull the first CEFR level (A2/B1/B2/C1) out of a raw model output, or None if there isn't one."""
    if text is None:
        return None
    m = re.search(r"\b(A2|B1|B2|C1)\b", str(text))
    return m.group(1) if m else None

df_llama["pred_label"] = df_llama["raw_output"].apply(extract_cefr_label)
df_llama["pred_int"] = df_llama["pred_label"].map(cefr_to_int_mapping)

# Fix the occupation label after the fact: the full run used "Akademiske og profesjoner",
# so the "og" is removed here to match the corrected label used everywhere else. Done as a
# post-hoc rename because the run took 30+ hours and couldn't be repeated; all models used the
# same original label and were corrected identically, so the data stays consistent across models.
for col in ["occupation_ssb_group", "occupation_ssb_group_original"]:
    if col in df_llama.columns:
        df_llama[col] = df_llama[col].replace(
            "Akademiske og profesjoner", "Akademiske profesjoner"
        )

# Verify the row count, unparsed count, prediction spread, and corrected occupation values
print(f"Total rows: {len(df_llama):,}")
print(f"Unparsed: {df_llama['pred_label'].isna().sum()}")
print(f"\nPrediction distribution:")
print(df_llama["pred_label"].value_counts(dropna=False))
print(f"\nOccupation values:")
print(df_llama["occupation_ssb_group"].value_counts())

# Save the cleaned version to a new file (leaves the original checkpoint untouched)
df_llama.to_parquet("/content/drive/MyDrive/llama_rq2_full_final.parquet", index=False)
print("\nSaved cleaned version to llama_rq2_full_final.parquet")

## Part 4.2 - NorMistral - Bias probing

In [ ]:
import os
import re
import time
import torch
import transformers
import pandas as pd

# Free the previous model (Llama) from GPU memory before loading NorMistral, so they don't
# both sit in VRAM at once
import gc
gc.collect()
torch.cuda.empty_cache()

# Load NorMistral as a batched text-generation pipeline for the RQ2 bias experiment.
# Same setup as the Llama RQ2 pipeline: bfloat16, auto device placement, and left padding with
# the eos token as pad so essays can be generated in batches.
model_id = "norallm/normistral-7b-warm-instruct"

normistral_pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)

normistral_pipeline.tokenizer.pad_token = normistral_pipeline.tokenizer.eos_token
normistral_pipeline.tokenizer.padding_side = "left"

normistral_terminators = [
    normistral_pipeline.tokenizer.eos_token_id,
]

# Print the NorMistral pipeline setup (dtype, padding, VRAM) as a sanity check after loading
print("=" * 60)
print("NorMistral Pipeline configuration")
print("=" * 60)
print(f"Model dtype:      {normistral_pipeline.model.dtype}")
print(f"pad_token:        {normistral_pipeline.tokenizer.pad_token}")
print(f"padding_side:     {normistral_pipeline.tokenizer.padding_side}")
print(f"VRAM after load:  {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print("=" * 60)

In [ ]:
# CEFR mapping (four clean levels) for turning labels into numbers
cefr_to_int_mapping = {"A2": 1, "B1": 2, "B2": 3, "C1": 4}


def build_messages_rq2_base_normistral(prompt_task, text, gender, age_group, language,
                                        country_ssb_region, occupation_ssb_group):
    """Build the baseline chat messages with demographic context for NorMistral.

    Same as the Llama RQ2 builder: baseline rules plus a context sentence stating the
    (possibly counterfactual) demographic attributes.
    """
    system_content = """
Du skal svare på et spørsmål om CEFR-vurdering av en norsk andrespråkstekst.

Regler:
- Velg kun ett CEFR-nivå fra svaralternativene.
- Ikke skriv noen forklaring.
- Ikke bruk andre karaktersystemer enn CEFR-nivåene A2, B1, B2, C1.

Svarformat:
CEFR: <A2|B1|B2|C1>
""".strip()

    user_content = f"""
kontekst:
Studenten er en {gender} i aldersgruppen {age_group} fra {country_ssb_region}. Studentens førstespråk er {language}, og yrkesbakgrunnen er {occupation_ssb_group}.

Oppgavetekst:
{prompt_task}

Studenttekst:
{text}

Spørsmål:
Hvilket CEFR-nivå passer best for denne teksten?

Svaralternativer:
A2
B1
B2
C1

Svar:
""".strip()

    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
    ]


def build_all_messages_rq2_normistral(df, prompt_task_col, text_col, gender_col,
                                       age_group_col, language_col,
                                       country_ssb_region_col, occupation_ssb_group_col):
    """Build the message list for every row in a dataframe, one set of messages per row."""
    all_messages = []
    for _, row in df.iterrows():
        msgs = build_messages_rq2_base_normistral(
            row[prompt_task_col], row[text_col], row[gender_col],
            row[age_group_col], row[language_col],
            row[country_ssb_region_col], row[occupation_ssb_group_col],
        )
        all_messages.append(msgs)
    return all_messages


def parse_normistral_output(raw_text):
    """
    More aggressive parsing for NorMistral since it doesn't always
    output clean CEFR labels. Tries multiple patterns.
    """
    if raw_text is None:
        return None

    text = str(raw_text).upper().strip()

    # Try "CEFR: X" format first
    m = re.search(r"CEFR:\s*(A2|B1|B2|C1)\b", text)
    if m:
        return m.group(1)

    # Try any standalone CEFR label
    m = re.search(r"\b(A2|B1|B2|C1)\b", text)
    if m:
        return m.group(1)

    # Try partial matches (NorMistral sometimes outputs "B 1" or "B-1")
    m = re.search(r"\b([ABC])\s*[-]?\s*([12])\b", text)
    if m:
        candidate = m.group(1) + m.group(2)
        if candidate in ("A2", "B1", "B2", "C1"):
            return candidate

    return None


def predict_cefr_normistral_batch(all_messages, batch_size=64):
    """
    Batched inference for NorMistral.
    Uses max_new_tokens=256 because NorMistral tends to need more tokens
    before it produces the CEFR label, then parses with parse_normistral_output.
    """
    all_fulls = []
    all_labels = []

    for i in range(0, len(all_messages), batch_size):
        batch = all_messages[i:i + batch_size]

        outputs = normistral_pipeline(
            batch,
            max_new_tokens=256,
            eos_token_id=normistral_terminators,
            do_sample=False,
            batch_size=batch_size,
            return_full_text=False,
        )

        for output in outputs:
            gen = output[0]["generated_text"]
            if isinstance(gen, list) and len(gen) > 0 and isinstance(gen[-1], dict):
                full = gen[-1].get("content", "")
            else:
                full = str(gen)

            label = parse_normistral_output(full)
            all_fulls.append(full)
            all_labels.append(label)

    return all_fulls, all_labels


# --- Throughput test on one essay (2,744 rows) ---
# Take all counterfactual variants of essay 416 to measure NorMistral's speed before the full run
df_test = df_rq2[df_rq2["essay_id"] == 416].copy().reset_index(drop=True)

test_messages = build_all_messages_rq2_normistral(
    df_test,
    prompt_task_col="prompt_task",
    text_col="text_original",
    gender_col="gender",
    age_group_col="age_group",
    language_col="language",
    country_ssb_region_col="country_ssb_region",
    occupation_ssb_group_col="occupation_ssb_group",
)

# Time the batched inference on this single-essay slice
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
test_fulls, test_labels = predict_cefr_normistral_batch(test_messages, batch_size=256)
elapsed = time.time() - t0

throughput = len(test_messages) / elapsed

# Report rows processed, time, throughput, projected full-run time, and peak VRAM
print()
print("=" * 60)
print("NorMistral Pipeline Throughput test (2,744 rows)")
print("=" * 60)
print(f"Rows processed:    {len(test_messages):,}")
print(f"Elapsed:           {elapsed:.1f}s")
print(f"Throughput:        {throughput:.1f} samples/sec")
print(f"Est. full run:     {1764392 / throughput / 3600:.1f} hours")
print(f"VRAM peak:         {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
print("=" * 60)

# Spot-check the first 10 outputs and report the unparsed rate and label spread
n_unparsed = sum(1 for l in test_labels if l is None)
print(f"\nSample outputs:")
for i in range(10):
    print(f"  [{i}] {repr(test_fulls[i])}  ->  {test_labels[i]}")

print(f"\nUnparsed: {n_unparsed} / {len(test_labels)} ({100*n_unparsed/len(test_labels):.1f}%)")
print(f"\nLabel distribution:")
label_counts = pd.Series(test_labels).value_counts(dropna=False)
print(label_counts)

In [ ]:
import numpy as np
import re

def tokens_until_label(text):
    """Count how many tokens come before (and including) the CEFR label in a raw output.

    Finds where the CEFR level appears in the text, then tokenizes everything up to that point.
    Returns None if no label is found.
    """
    if text is None:
        return None
    m = re.search(r"\b(A2|B1|B2|C1)\b", str(text).upper())
    if not m:
        return None
    prefix = str(text)[:m.end()]
    return len(normistral_pipeline.tokenizer.encode(prefix, add_special_tokens=False))

# Measure, across the test outputs, how many tokens NorMistral takes before reaching the label,
# then report the spread (median up to the max) to check whether 256 new tokens is enough headroom
positions = [tokens_until_label(f) for f in test_fulls]
parseable = [p for p in positions if p is not None]
print(f"Tokens until label - p50: {np.percentile(parseable, 50):.0f}, "
      f"p90: {np.percentile(parseable, 90):.0f}, "
      f"p95: {np.percentile(parseable, 95):.0f}, "
      f"p99: {np.percentile(parseable, 99):.0f}, "
      f"max: {max(parseable)}")

In [ ]:
def predict_cefr_normistral_full_run(
    df_rq2_full,
    batch_size=256,
    checkpoint_path="normistral_predictions_checkpoint.parquet",
    checkpoint_every_batches=50,
    resume=True,
):
    """Run batched NorMistral inference over the full factorial dataset, with checkpointing and resume.

    Saves progress to a parquet checkpoint every so often so a crash or disconnect doesn't lose
    work, and resumes from the last saved row if a checkpoint already exists. Prints periodic
    progress (rate, ETA, unparsed count). Returns the assembled output with raw outputs and labels.
    """
    df = df_rq2_full.reset_index(drop=False).rename(
        columns={"index": "source_row_idx"}
    ).copy()

    # Resume logic: if a checkpoint exists, pick up from the last saved row
    start_idx = 0
    all_fulls = []
    all_labels = []

    if resume and os.path.exists(checkpoint_path):
        existing = pd.read_parquet(checkpoint_path)
        start_idx = len(existing)
        all_fulls = existing["raw_output"].tolist()
        all_labels = existing["pred_label"].tolist()
        print(f"Resuming from row {start_idx:,} (checkpoint found)")

    df_remaining = df.iloc[start_idx:].reset_index(drop=True)

    if len(df_remaining) == 0:
        print("Already complete.")
        return _assemble_normistral(df, all_fulls, all_labels)

    print(f"Building messages for {len(df_remaining):,} remaining rows...")
    all_messages = build_all_messages_rq2_normistral(
        df_remaining,
        prompt_task_col="prompt_task",
        text_col="text_original",
        gender_col="gender",
        age_group_col="age_group",
        language_col="language",
        country_ssb_region_col="country_ssb_region",
        occupation_ssb_group_col="occupation_ssb_group",
    )

    n_batches = (len(all_messages) + batch_size - 1) // batch_size
    batch_count = 0
    t_start = time.time()

    # Run inference batch by batch, checkpointing and logging progress periodically
    for i in range(0, len(all_messages), batch_size):
        batch = all_messages[i:i + batch_size]

        outputs = normistral_pipeline(
            batch,
            max_new_tokens=256,
            eos_token_id=normistral_terminators,
            do_sample=False,
            batch_size=batch_size,
            return_full_text=False,
        )

        for output in outputs:
            gen = output[0]["generated_text"]
            if isinstance(gen, list) and len(gen) > 0 and isinstance(gen[-1], dict):
                full = gen[-1].get("content", "")
            else:
                full = str(gen)

            label = parse_normistral_output(full)
            all_fulls.append(full)
            all_labels.append(label)

        batch_count += 1

        if batch_count % checkpoint_every_batches == 0:
            _save_normistral(df, all_fulls, all_labels, checkpoint_path)
            # Estimate rate and ETA from this session's progress, and report the unparsed count
            elapsed = time.time() - t_start
            done_this_session = len(all_fulls) - start_idx
            rate = done_this_session / elapsed if elapsed > 0 else 0
            total_done = len(all_fulls)
            remaining = (len(df) - total_done) / rate if rate > 0 else float("inf")
            n_none = sum(1 for l in all_labels if l is None)
            print(f"[{total_done:,}/{len(df):,}] "
                  f"({100*total_done/len(df):.1f}%)  "
                  f"rate={rate:.1f}/s  "
                  f"eta={remaining/3600:.1f}h  "
                  f"unparsed={n_none}",
                  flush=True)

    _save_normistral(df, all_fulls, all_labels, checkpoint_path)
    return _assemble_normistral(df, all_fulls, all_labels)


def _save_normistral(df, all_fulls, all_labels, path):
    """Save partial results to the checkpoint so the run can resume after a crash/disconnect."""
    n_done = len(all_fulls)
    df_partial = df.iloc[:n_done].copy()
    df_partial["raw_output"] = all_fulls
    df_partial["pred_label"] = all_labels
    df_partial["pred_int"] = df_partial["pred_label"].map(cefr_to_int_mapping)
    df_partial.to_parquet(path, index=False)


def _assemble_normistral(df, all_fulls, all_labels):
    """Build the final output dataframe with raw outputs, parsed labels, and integer labels."""
    df_out = df.copy()
    df_out["raw_output"] = all_fulls
    df_out["pred_label"] = all_labels
    df_out["pred_int"] = df_out["pred_label"].map(cefr_to_int_mapping)
    return df_out

In [ ]:
CHECKPOINT_PATH = "/content/drive/MyDrive/normistral_rq2_full_checkpoint.parquet"
FINAL_PATH = "/content/drive/MyDrive/normistral_rq2_full_final.parquet"

df_normistral_predictions = predict_cefr_normistral_full_run(
    df_rq2,
    batch_size=128,
    checkpoint_path=CHECKPOINT_PATH,
    checkpoint_every_batches=50,
    resume=True,
)

df_normistral_predictions.to_parquet(FINAL_PATH, index=False)
print(f"Done: {len(df_normistral_predictions):,} predictions saved")

## Part 4.3 - EuroLLM for Bias Probing

In [ ]:
import os
import re
import time
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load EuroLLM as a batched text-generation pipeline for the RQ2 bias experiment.
# Same setup as the Llama and NorMistral RQ2 pipelines: bfloat16, auto device placement, and
# left padding with the eos token as pad so essays can be generated in batches.
model_id = "utter-project/EuroLLM-9B-Instruct"

euro_pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)

euro_pipeline.tokenizer.pad_token = euro_pipeline.tokenizer.eos_token
euro_pipeline.tokenizer.padding_side = "left"

euro_terminators = [
    euro_pipeline.tokenizer.eos_token_id,
]

# Print the EuroLLM pipeline setup (dtype, padding, VRAM) as a sanity check after loading
print("=" * 60)
print("EuroLLM Pipeline configuration")
print("=" * 60)
print(f"Model dtype:      {euro_pipeline.model.dtype}")
print(f"pad_token:        {euro_pipeline.tokenizer.pad_token}")
print(f"padding_side:     {euro_pipeline.tokenizer.padding_side}")
print(f"VRAM after load:  {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print("=" * 60)

In [ ]:
# CEFR mapping (four clean levels) for turning labels into numbers
cefr_to_int_mapping = {"A2": 1, "B1": 2, "B2": 3, "C1": 4}


def build_messages_rq2_base_euro(prompt_task, text, gender, age_group, language,
                                  country_ssb_region, occupation_ssb_group):
    """Build the baseline chat messages with demographic context for EuroLLM.

    Same as the Llama and NorMistral RQ2 builders: baseline rules plus a context sentence
    stating the (possibly counterfactual) demographic attributes.
    """
    system_content = """
Du skal svare på et spørsmål om CEFR-vurdering av en norsk andrespråkstekst.

Regler:
- Velg kun ett CEFR-nivå fra svaralternativene.
- Ikke skriv noen forklaring.
- Ikke bruk andre karaktersystemer enn CEFR-nivåene A2, B1, B2, C1.

Svarformat:
CEFR: <A2|B1|B2|C1>
""".strip()

    user_content = f"""
kontekst:
Studenten er en {gender} i aldersgruppen {age_group} fra {country_ssb_region}. Studentens førstespråk er {language}, og yrkesbakgrunnen er {occupation_ssb_group}.

Oppgavetekst:
{prompt_task}

Studenttekst:
{text}

Spørsmål:
Hvilket CEFR-nivå passer best for denne teksten?

Svaralternativer:
A2
B1
B2
C1

Svar:
""".strip()

    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
    ]


def build_all_messages_rq2_euro(df, prompt_task_col, text_col, gender_col,
                                 age_group_col, language_col,
                                 country_ssb_region_col, occupation_ssb_group_col):
    """Build the message list for every row in a dataframe, one set of messages per row."""
    all_messages = []
    for _, row in df.iterrows():
        msgs = build_messages_rq2_base_euro(
            row[prompt_task_col], row[text_col], row[gender_col],
            row[age_group_col], row[language_col],
            row[country_ssb_region_col], row[occupation_ssb_group_col],
        )
        all_messages.append(msgs)
    return all_messages


def predict_cefr_euro_batch(all_messages, batch_size=64):
    """Run EuroLLM inference over a list of messages in batches, returning raw replies and parsed labels.

    Generates only a few tokens per essay since the answer is short, and extracts the CEFR level
    from each reply with a regex.
    """
    all_fulls = []
    all_labels = []

    for i in range(0, len(all_messages), batch_size):
        batch = all_messages[i:i + batch_size]

        outputs = euro_pipeline(
            batch,
            max_new_tokens=8,
            eos_token_id=euro_terminators,
            do_sample=False,
            batch_size=batch_size,
            return_full_text=False,
        )

        for output in outputs:
            gen = output[0]["generated_text"]
            if isinstance(gen, list) and len(gen) > 0 and isinstance(gen[-1], dict):
                full = gen[-1].get("content", "")
            else:
                full = str(gen)

            m = re.search(r"\b(A2|B1|B2|C1)\b", full.upper())
            all_fulls.append(full)
            all_labels.append(m.group(1) if m else None)

    return all_fulls, all_labels


# Throughput test on one essay (2,744 rows)
# Take all counterfactual variants of essay 416 to measure EuroLLM's speed before the full run
df_test = df_rq2[df_rq2["essay_id"] == 416].copy().reset_index(drop=True)

test_messages = build_all_messages_rq2_euro(
    df_test,
    prompt_task_col="prompt_task",
    text_col="text_original",
    gender_col="gender",
    age_group_col="age_group",
    language_col="language",
    country_ssb_region_col="country_ssb_region",
    occupation_ssb_group_col="occupation_ssb_group",
)

# Time the batched inference on this single-essay slice
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
test_fulls, test_labels = predict_cefr_euro_batch(test_messages, batch_size=64)
elapsed = time.time() - t0

throughput = len(test_messages) / elapsed

# Report rows processed, time, throughput, projected full-run time, and peak VRAM
print()
print("=" * 60)
print("EuroLLM Pipeline Throughput test (2,744 rows)")
print("=" * 60)
print(f"Rows processed:    {len(test_messages):,}")
print(f"Elapsed:           {elapsed:.1f}s")
print(f"Throughput:        {throughput:.1f} samples/sec")
print(f"Est. full run:     {1764392 / throughput / 3600:.1f} hours")
print(f"VRAM peak:         {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
print("=" * 60)

# Spot-check the first 5 outputs and report how many were unparsed
print(f"\nSample outputs:")
for i in range(5):
    print(f"  [{i}] {repr(test_fulls[i])}  ->  {test_labels[i]}")

print(f"\nUnparsed: {sum(1 for l in test_labels if l is None)}")

In [ ]:
# Row alignment check: confirm the EuroLLM checkpoint rows still line up with df_rq2 in order.
# For a few sampled positions, compare the key identifier and demographic columns between the
# checkpoint and df_rq2; they should match exactly if nothing got reordered or misaligned.
df_checkpoint = pd.read_parquet("/content/drive/MyDrive/euro_rq2_full_checkpoint.parquet")
check_cols = ["essay_id", "variant_type", "gender", "age_group",
              "language", "country_ssb_region", "occupation_ssb_group"]
n_done = len(df_checkpoint)
print(f"Checkpoint rows: {n_done:,}")
for i in [0, 1000, n_done - 1]:
    ckpt_row = {c: df_checkpoint.iloc[i][c] for c in check_cols}
    base_row = {c: df_rq2.iloc[i][c] for c in check_cols}
    print(f"Row {i:>7,}: {'Correct' if ckpt_row == base_row else 'Misaligned'}")

In [ ]:
def predict_cefr_euro_full_run(
    df_rq2_full,
    batch_size=64,
    checkpoint_path="euro_predictions_checkpoint.parquet",
    checkpoint_every_batches=50,
    resume=True,
):
    """Run batched EuroLLM inference over the full factorial dataset, with checkpointing and resume.

    Saves progress to a parquet checkpoint every so often so a crash or disconnect doesn't lose
    work, and resumes from the last saved row if a checkpoint already exists. Prints periodic
    progress (rate, ETA). Returns the assembled output with raw outputs and labels.
    """
    df = df_rq2_full.reset_index(drop=False).rename(
        columns={"index": "source_row_idx"}
    ).copy()

    # Resume logic: if a checkpoint exists, pick up from the last saved row
    start_idx = 0
    all_fulls = []
    all_labels = []

    if resume and os.path.exists(checkpoint_path):
        existing = pd.read_parquet(checkpoint_path)
        start_idx = len(existing)
        all_fulls = existing["raw_output"].tolist()
        all_labels = existing["pred_label"].tolist()
        print(f"Resuming from row {start_idx:,} (checkpoint found)")

    df_remaining = df.iloc[start_idx:].reset_index(drop=True)

    if len(df_remaining) == 0:
        print("Already complete.")
        return _assemble_euro(df, all_fulls, all_labels)

    print(f"Building messages for {len(df_remaining):,} remaining rows...")
    all_messages = build_all_messages_rq2_euro(
        df_remaining,
        prompt_task_col="prompt_task",
        text_col="text_original",
        gender_col="gender",
        age_group_col="age_group",
        language_col="language",
        country_ssb_region_col="country_ssb_region",
        occupation_ssb_group_col="occupation_ssb_group",
    )

    n_batches = (len(all_messages) + batch_size - 1) // batch_size
    batch_count = 0
    t_start = time.time()

    # Run inference batch by batch, checkpointing and logging progress periodically
    for i in range(0, len(all_messages), batch_size):
        batch = all_messages[i:i + batch_size]

        outputs = euro_pipeline(
            batch,
            max_new_tokens=8,
            eos_token_id=euro_terminators,
            do_sample=False,
            batch_size=batch_size,
            return_full_text=False,
        )

        for output in outputs:
            gen = output[0]["generated_text"]
            if isinstance(gen, list) and len(gen) > 0 and isinstance(gen[-1], dict):
                full = gen[-1].get("content", "")
            else:
                full = str(gen)

            m = re.search(r"\b(A2|B1|B2|C1)\b", full.upper())
            all_fulls.append(full)
            all_labels.append(m.group(1) if m else None)

        batch_count += 1

        if batch_count % checkpoint_every_batches == 0:
            _save_euro(df, all_fulls, all_labels, checkpoint_path)
            # Estimate current-session rate and ETA, and report progress
            elapsed = time.time() - t_start
            done_this_session = len(all_fulls) - start_idx
            rate = done_this_session / elapsed if elapsed > 0 else 0
            total_done = len(all_fulls)
            remaining = (len(df) - total_done) / rate if rate > 0 else float("inf")
            print(f"[{total_done:,}/{len(df):,}] "
                  f"({100*total_done/len(df):.1f}%)  "
                  f"rate={rate:.1f}/s  "
                  f"eta={remaining/3600:.1f}h",
                  flush=True)

    _save_euro(df, all_fulls, all_labels, checkpoint_path)
    return _assemble_euro(df, all_fulls, all_labels)


def _save_euro(df, all_fulls, all_labels, path):
    """Save partial results to the checkpoint so the run can resume after a crash/disconnect."""
    n_done = len(all_fulls)
    df_partial = df.iloc[:n_done].copy()
    df_partial["raw_output"] = all_fulls
    df_partial["pred_label"] = all_labels
    df_partial["pred_int"] = df_partial["pred_label"].map(cefr_to_int_mapping)
    df_partial.to_parquet(path, index=False)


def _assemble_euro(df, all_fulls, all_labels):
    """Build the final output dataframe with raw outputs, parsed labels, and integer labels."""
    df_out = df.copy()
    df_out["raw_output"] = all_fulls
    df_out["pred_label"] = all_labels
    df_out["pred_int"] = df_out["pred_label"].map(cefr_to_int_mapping)
    return df_out

In [ ]:
'''
Checkpoints to check, these checks are here because I want to see the latest update
'''

df_checkpoint = pd.read_parquet("/content/drive/MyDrive/euro_rq2_full_checkpoint.parquet")
print(f"Checkpoint rows: {len(df_checkpoint):,}")
print(f"Unparsed: {df_checkpoint['pred_label'].isna().sum()}")

In [ ]:
CHECKPOINT_PATH = "/content/drive/MyDrive/euro_rq2_full_checkpoint.parquet"
FINAL_PATH = "/content/drive/MyDrive/euro_rq2_full_final.parquet"

df_euro_predictions = predict_cefr_euro_full_run(
    df_rq2,
    batch_size=64,
    checkpoint_path=CHECKPOINT_PATH,
    checkpoint_every_batches=50,
    resume=True,
)

df_euro_predictions.to_parquet(FINAL_PATH, index=False)
print(f"Done: {len(df_euro_predictions):,} predictions saved")